In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import re
import time
import pickle
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score
from xgboost import XGBClassifier

MINIMAX_API_KEY  = os.getenv("MINIMAX_API_KEY", "")
MINIMAX_URL      = "https://api.minimaxi.com/v1/chat/completions"
SAVE_PATH_TEST   = "../data/text_analyst_results_test_matched_sample1.pkl"
SAVE_PATH_TRAIN  = "../data/text_analyst_results_train_matched_sample1.pkl"

# Logit / sigmoid helpers (used in logit-space fusion)
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=float)))


# Generic LLM call helper (used by green_llm_review and other agents)
def call_llm(prompt, max_tokens=300, temperature=0.2):
    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": max_tokens, "temperature": temperature},
        timeout=30,
    )
    return resp.json()['choices'][0]['message']['content']


## 1. Data Loading

In [ ]:
df = pd.read_csv('../loan_default.csv', low_memory=False)

# label (0/1) and desc are already present in the cleaned CSV
print(f"Loaded: {len(df)} records, default rate: {df['label'].mean():.1%}")
print(f"Grade distribution:")
print(df['grade'].value_counts().sort_index())

## 2. Train / Test Split

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train: {len(train)}, default rate: {train['label'].mean():.1%}")
print(f"Test:  {len(test)},  default rate: {test['label'].mean():.1%}")

## 3. Numeric Feature Definitions

In [ ]:
NUM_FEATURES = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc',
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies'
]

## 4. White Agent — Text Analyst

In [ ]:
def text_analyst_agent(desc_text: str) -> dict:
    """Score borrower description; returns continuous text_risk_score (0-1)."""
    desc_text = re.sub(r'<[^>]+>', ' ', str(desc_text)).strip()[:500]

    word_count  = len(desc_text.split())
    has_numbers = bool(re.search(r'\$[\d,]+|\d+%|\d+ months?|\d+ years?', desc_text))
    has_plan    = bool(re.search(r'will pay|plan to|intend|commit|currently|stable', desc_text, re.I))
    has_stress  = bool(re.search(r'emergency|urgent|desperate|behind|struggling|need immediately', desc_text, re.I))

    prompt = (
        "You are evaluating loan applications from Grade C-G borrowers on LendingClub.\n"
        "These borrowers already have elevated credit risk — roughly 15-25% will default.\n"
        "Your text_risk_score should reflect this: a TYPICAL C-G borrower with a vague description "
        "should score around 0.40-0.50, not 0.20-0.30.\n\n"
        f"Description: {desc_text}\n\n"
        "Scoring anchors for text_risk_score:\n"
        "  0.05-0.20: unusually strong evidence — specific dollar amounts, concrete timeline, "
        "verified stable income, explicit low-rate refinancing with repayment math\n"
        "  0.20-0.35: genuinely positive signals — mentions stable job, clear single purpose, "
        "no stress language, some concrete detail\n"
        "  0.35-0.55: average C-G borrower — vague purpose, standard language, "
        "no strong signals in either direction; debt consolidation alone falls here\n"
        "  0.55-0.70: mild concern — multiple debts mentioned without plan, "
        "vague about income, hints of urgency or pressure\n"
        "  0.70-0.90: clear stress signals — urgent tone, behind on bills, "
        "no income mentioned, consolidating multiple problem debts with no plan\n"
        "  0.90-1.00: severe distress — explicitly behind on payments, emergency cash need, "
        "no repayment plan whatsoever\n\n"
        "IMPORTANT: Do NOT default to 0.2-0.3 for all descriptions. "
        "If you are uncertain, score 0.40-0.50. "
        "Debt consolidation by itself (without concrete repayment details) is 0.40-0.50, not 0.20-0.30.\n\n"
        "Output a single JSON object and nothing else:\n"
        '{"text_risk_score": <float 0.0-1.0>,\n'
        ' "confidence": <float 0.0-1.0>,\n'
        ' "risk_signals": [<up to 3 quoted phrases>],\n'
        ' "protective_signals": [<up to 3 quoted phrases>],\n'
        ' "reasoning": "<one sentence>"}\n\n'
        "Confidence guide:\n"
        "  0.85-0.95: specific dollar amounts, timelines, or explicit repayment plans\n"
        "  0.65-0.84: some concrete details\n"
        "  0.50-0.64: vague or only 1-2 sentences\n"
        "  0.40-0.49: extremely short or nearly no information"
    )

    for attempt in range(3):
        try:
            resp = requests.post(
                MINIMAX_URL,
                headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
                json={"model": "MiniMax-Text-01",
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": 300, "temperature": 0.3},
                timeout=30
            )
            raw = resp.json()['choices'][0]['message']['content']
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if not match:
                raise ValueError(f"No JSON: {raw[:100]}")
            result = json.loads(match.group(0))

            # Calibrate confidence by text quality
            quality  = (min(word_count / 100, 0.3) + (0.2 if has_numbers else 0)
                        + (0.15 if has_plan else 0) + (0.1 if has_stress else 0))
            raw_conf = result.get('confidence', 0.5)
            result['confidence'] = round(min(max(0.6 * raw_conf + 0.4 * (0.4 + quality), 0.4), 0.95), 3)
            result['text_risk_score'] = round(min(max(float(result.get('text_risk_score', 0.45)), 0.0), 1.0), 3)
            return result
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

## 5. Run Text Analysis (Grades C–G)

In [ ]:
def run_text_analysis(df_subset, save_path):
    """Run text_analyst_agent on df_subset with resume support."""
    try:
        with open(save_path, 'rb') as f:
            results = pickle.load(f)
        done_idx  = {r['idx'] for r in results}
        remaining = len(df_subset) - len(done_idx)
        print(f"  Resuming {os.path.basename(save_path)}: {len(results)} done, {remaining} remaining")
    except FileNotFoundError:
        results, done_idx = [], set()
        print(f"  Starting fresh: {len(df_subset)} records")

    errors = []
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        if idx in done_idx:
            continue
        try:
            result = text_analyst_agent(row['desc'])
            result['idx'] = idx
            results.append(result)
            if len(results) % 50 == 0:
                with open(save_path, 'wb') as f:
                    pickle.dump(results, f)
        except Exception as e:
            errors.append({'idx': idx, 'error': str(e)})
            time.sleep(1)
        time.sleep(0.1)

    with open(save_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"  Done: {len(results)}, failed: {len(errors)}")
    return results


# C-G grades only; cap API calls with TEXT_SAMPLE_SIZE
TEXT_SAMPLE_SIZE = 2000  # increase for a production run

cg_test  = test[test['grade'].isin(['C', 'D', 'E', 'F', 'G'])]
cg_train = train[train['grade'].isin(['C', 'D', 'E', 'F', 'G'])]

target_test  = (cg_test.sample(n=min(TEXT_SAMPLE_SIZE // 4, len(cg_test)),   random_state=42)
                if len(cg_test)  > TEXT_SAMPLE_SIZE // 4  else cg_test).copy()
target_train = (cg_train.sample(n=min(TEXT_SAMPLE_SIZE,       len(cg_train)), random_state=42)
                if len(cg_train) > TEXT_SAMPLE_SIZE         else cg_train).copy()

print(f"C-G train pool: {len(cg_train)}  →  text-analyze: {len(target_train)}")
print(f"C-G test  pool: {len(cg_test)}   →  text-analyze: {len(target_test)}")

if os.getenv("SKIP_TEXT_ANALYST", "0") != "1":
    print("\n--- Running on TEST ---")
    run_text_analysis(target_test, SAVE_PATH_TEST)
    print("\n--- Running on TRAIN ---")
    run_text_analysis(target_train, SAVE_PATH_TRAIN)
else:
    print("\n[SKIP_TEXT_ANALYST=1]  reusing existing pkl caches; no new LLM calls")

## 6. Build Fused Subset DataFrame

In [112]:
def load_text_subset(df, save_path):
    with open(save_path, 'rb') as f:
        results = pickle.load(f)
    rdf = pd.DataFrame(results).set_index('idx')

    # Backward compat: old format used 'risk_label' (binary)
    if 'text_risk_score' not in rdf.columns:
        rdf['text_risk_score'] = rdf['risk_label'].astype(float)

    sub = df.join(rdf[['text_risk_score', 'confidence']], how='left')
    sub = sub.rename(columns={'confidence': 'text_confidence'})
    return sub.dropna(subset=['text_risk_score', 'text_confidence'])


target_test  = test[test['grade'].isin(['C', 'D', 'E', 'F', 'G'])].copy()
target_train = train[train['grade'].isin(['C', 'D', 'E', 'F', 'G'])].copy()

subset_train = load_text_subset(target_train, SAVE_PATH_TRAIN)
subset_test  = load_text_subset(target_test,  SAVE_PATH_TEST)

print(f"subset_train: {len(subset_train)} records, default rate: {subset_train['label'].mean():.2f}")
print(f"subset_test:  {len(subset_test)}  records, default rate: {subset_test['label'].mean():.2f}")
print(f"Raw text_risk_score (train): mean={subset_train['text_risk_score'].mean():.3f}, "
      f"std={subset_train['text_risk_score'].std():.3f}")

# ── Per-grade z-score normalization (train stats → applied to both) ─────────
# Linear stretch fails because LLM underestimates absolute risk levels.
# Z-score normalization uses rank signal within each grade — if the LLM
# correctly orders borrowers by relative risk, this preserves that signal
# while re-centering each grade at 0.5 (neutral).
grade_norm_stats = {}
for grade, grp in subset_train.groupby('grade'):
    m = grp['text_risk_score'].mean()
    s = grp['text_risk_score'].std()
    grade_norm_stats[grade] = (m, s if s > 0.01 else 1.0)

def _normalize(score, grade):
    if grade not in grade_norm_stats:
        return 0.5
    m, s = grade_norm_stats[grade]
    z = (score - m) / s
    return round(min(max(0.5 + z * 0.15, 0.05), 0.95), 4)

subset_train['text_risk_score'] = subset_train.apply(
    lambda r: _normalize(r['text_risk_score'], r['grade']), axis=1)
subset_test['text_risk_score'] = subset_test.apply(
    lambda r: _normalize(r['text_risk_score'], r['grade']), axis=1)

print(f"Normalized text_risk_score (train): mean={subset_train['text_risk_score'].mean():.3f}, "
      f"std={subset_train['text_risk_score'].std():.3f}, "
      f">0.5: {(subset_train['text_risk_score']>0.5).mean():.1%}")

# text_stats from TRAIN only (no test leakage)
text_stats = {}
for grade, grp in subset_train.groupby('grade'):
    text_stats[grade] = {
        'text_risk1_rate':     round(float((grp['text_risk_score'] > 0.5).mean()), 3),
        'actual_default_rate': round(float(grp['label'].mean()), 3),
        'avg_confidence':      round(float(grp['text_confidence'].mean()), 3),
        'avg_risk_score':      round(float(grp['text_risk_score'].mean()), 3),
        'n':                   len(grp),
    }
print("\nPer-grade text stats (train, normalized):")
print(pd.DataFrame(text_stats).T)


subset_train: 552 records, default rate: 0.24
subset_test:  34  records, default rate: 0.26
Raw text_risk_score (train): mean=0.312, std=0.086
Normalized text_risk_score (train): mean=0.498, std=0.142, >0.5: 28.1%

Per-grade text stats (train, normalized):
   text_risk1_rate  actual_default_rate  avg_confidence  avg_risk_score      n
C            0.198                0.201           0.675           0.498  283.0
D            0.281                0.231           0.668           0.497  160.0
E            0.582                0.328           0.699           0.499   67.0
F            0.306                0.444           0.693           0.500   36.0
G            0.667                0.000           0.713           0.500    6.0


---
## Multi-Agent System

**Green Agent** owns the quantitative pipeline: trains XGBoost, evaluates fusion, PSI drift detection, precision gate. White Agents cannot override these decisions.

**Five White Agents** — the system shifts the narrative from *predicting better* to *when to trust the text*:

| # | Agent | LLM policy |
|---|---|---|
| 1 | Text Analyst | Always — free text requires LLM interpretation |
| 2 | Feature Strategist | Conditional — only when grade signals are mixed/ambiguous |
| 3 | Subgroup Advocate | Never — purely quantitative checks (AUC, precision, recall) |
| 4 | Arbitrator | On-demand — only when Advocate vetoes a Strategist proposal |
| 5 | Reporter | Per fuzzy-zone case — explains and audits final predictions |

**Iterative loop flow:**  
Green evaluates → Advocate checks → *if veto*: Strategist proposes + Arbitrator mediates → *else*: Strategist updates directly → repeat until convergence

#### Green Agent

In [ ]:
def call_llm(prompt, max_tokens=300, temperature=0.2):
    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": max_tokens, "temperature": temperature},
        timeout=30,
    )
    return resp.json()['choices'][0]['message']['content']

class GreenAgent:
    """
    Execution environment. Owns: baseline model, subgroup diagnostics,
    PSI shift detection, fusion evaluation, threshold calibration, precision gate.
    White Agents cannot override any of these decisions.

    Reflexion (cognitive self-evolving planning, MultiAgentBench A.12):
      Each iteration Green generates an expectation BEFORE evaluating the
      strategy, then reflects on expected-vs-actual AFTER. Errors accumulate
      in self.experience and feed back into adaptive validation thresholds,
      Green's own counter-proposal, and the LLM review prompt.
    """

    def __init__(self, trained_model, num_features):
        self.model           = trained_model
        self.num_features    = num_features
        self.experience      = []     # list of {iter, strategy, expectation, actual, errors}
        self.gating_fn       = None   # learned per-row text-weight gate (Tier-A); None → use grade_weights
        self.gating_summary  = None   # diagnostic info from train_gating()

    # ── Reflexion infrastructure ──────────────────────────────────────────
    def _summarize_experience(self, last_k=3) -> str:
        if not self.experience:
            return "(no prior iterations)"
        lines = []
        for rec in self.experience[-last_k:]:
            i      = rec['iter']
            exp_da = rec['expectation'].get('expected_overall_delta_auc', 0.0)
            act_da = rec['actual'].get('actual_overall_delta_auc',        0.0)
            err_ms = rec['errors'].get('err_mean_shift',                  0.0)
            lines.append(
                f"  iter {i}: predicted ΔAUC={exp_da:+.4f}, actual={act_da:+.4f} "
                f"(err={act_da - exp_da:+.4f}); mean-shift err={err_ms:+.4f}"
            )
        return "\n".join(lines)

    def generate_expectation(self, strategy, prev_result=None, iteration=0) -> dict:
        """
        Predict ΔAUC, mean_shift, per-grade deltas BEFORE running evaluate_fusion.
        Cold start (iter 0, no history) returns a neutral prior without LLM call.
        On LLM failure, falls back to extrapolating the trajectory mean.
        """
        if iteration == 0 and not self.experience:
            return {
                'expected_overall_delta_auc': 0.0,
                'expected_mean_shift':        0.02,
                'expected_per_grade_delta':   {g: 0.0 for g in strategy.get('grade_weights', {})},
                'confidence':                 0.3,
                'rationale':                  'cold start; no prior to anchor on',
                'llm_called':                 False,
            }

        prev_summary = ""
        if prev_result is not None:
            prev_summary = (
                f"Previous iteration: baseline AUC={prev_result['overall_baseline_auc']:.4f}, "
                f"fused AUC={prev_result['overall_fused_auc']:.4f}\n"
                f"Per-grade deltas: "
                f"{json.dumps({g: round(s['delta'], 4) for g, s in prev_result['grade_auc'].items()})}\n"
            )

        prompt = (
            f"You are the Green Agent's planning module — iteration {iteration}.\n"
            "Predict what will happen if this fusion strategy is applied. Anchor on the trajectory below; "
            "if past predictions overshot, dampen.\n\n"
            f"Strategy: grade_weights={json.dumps(strategy.get('grade_weights', {}))}, "
            f"conf_threshold={strategy.get('conf_threshold', 0.65)}\n\n"
            f"{prev_summary}"
            f"Your past prediction trajectory (recent):\n{self._summarize_experience(3)}\n\n"
            "Predict (be quantitative):\n"
            "  - expected_overall_delta_auc: fused AUPRC minus baseline AUPRC (primary signal), signed float ~[-0.03, +0.05]\n"
            "  - expected_mean_shift: |fused-baseline| mean, positive float ~[0, 0.15]\n"
            "  - expected_per_grade_delta: dict mapping each grade in the strategy to a signed delta\n"
            "  - confidence: 0-1\n"
            "  - rationale: one sentence\n\n"
            "CRITICAL OUTPUT FORMAT: respond with ONLY the JSON object. No reasoning, no markdown code fences, no commentary. Your response must start with { and end with }. Any text outside the JSON braces will break the parser.\n\n" + 'Output only JSON: '
            '{"expected_overall_delta_auc": <float>, "expected_mean_shift": <float>, '
            '"expected_per_grade_delta": {<grade>: <float>, ...}, "confidence": <float>, "rationale": "<text>"}'
        )

        try:
            raw     = call_llm(prompt, max_tokens=800, temperature=0.2)
            cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip())
            cleaned = re.sub(r"\s*```$", "", cleaned.strip())
            parsed  = json.loads(re.search(r"\{.*\}", cleaned, re.DOTALL).group(0))
            return {
                'expected_overall_delta_auc': float(parsed.get('expected_overall_delta_auc', 0.0)),
                'expected_mean_shift':        float(parsed.get('expected_mean_shift', 0.02)),
                'expected_per_grade_delta':   {g: float(v) for g, v in
                                               parsed.get('expected_per_grade_delta', {}).items()},
                'confidence':                 float(parsed.get('confidence', 0.5)),
                'rationale':                  str(parsed.get('rationale', '')),
                'llm_called':                 True,
                'llm_raw':                    raw,
            }
        except Exception as e:
            past_da = [rec['actual'].get('actual_overall_delta_auc', 0.0) for rec in self.experience]
            past_ms = [rec['actual'].get('actual_mean_shift',        0.02) for rec in self.experience]
            return {
                'expected_overall_delta_auc': float(np.mean(past_da)) if past_da else 0.0,
                'expected_mean_shift':        float(np.mean(past_ms)) if past_ms else 0.02,
                'expected_per_grade_delta':   {g: 0.0 for g in strategy.get('grade_weights', {})},
                'confidence':                 0.4,
                'rationale':                  f'heuristic fallback: {e}',
                'llm_called':                 False,
            }

    def reflect(self, expectation, actual_result, strategy) -> dict:
        """
        Compare expected vs actual; append to self.experience; return error summary.
        """
        baseline = actual_result.get('baseline_preds', np.array([]))
        fused    = actual_result.get('fused_preds',    np.array([]))
        actual_mean_shift = float(np.abs(fused - baseline).mean()) if len(baseline) and len(fused) else 0.0
        actual_delta_auc  = float(actual_result['overall_fused_auc']
                                   - actual_result['overall_baseline_auc'])
        actual_per_grade  = {g: float(s['delta']) for g, s in actual_result.get('grade_auc', {}).items()}

        actual = {
            'actual_overall_delta_auc': actual_delta_auc,
            'actual_mean_shift':        actual_mean_shift,
            'actual_per_grade_delta':   actual_per_grade,
        }

        exp_grade = expectation.get('expected_per_grade_delta', {})
        err_delta = actual_delta_auc  - expectation.get('expected_overall_delta_auc', 0.0)
        err_ms    = actual_mean_shift - expectation.get('expected_mean_shift',        0.0)
        err_grade = {g: actual_per_grade.get(g, 0.0) - exp_grade.get(g, 0.0)
                     for g in set(actual_per_grade) | set(exp_grade)}
        surprise  = abs(err_delta) + abs(err_ms)

        errors = {
            'err_delta_auc':  float(err_delta),
            'err_mean_shift': float(err_ms),
            'err_per_grade':  err_grade,
            'surprise':       float(surprise),
        }

        self.experience.append({
            'iter':        len(self.experience),
            'strategy':    {k: v for k, v in strategy.items() if k != 'source'},
            'expectation': expectation,
            'actual':      actual,
            'errors':      errors,
        })

        bias_delta = float(np.mean([r['errors']['err_delta_auc']  for r in self.experience]))
        bias_ms    = float(np.mean([r['errors']['err_mean_shift'] for r in self.experience]))
        return {'errors': errors, 'bias_delta': bias_delta, 'bias_ms': bias_ms,
                'n_history': len(self.experience)}

    def adaptive_validation_thresholds(self) -> dict:
        """
        Derive context-aware thresholds from experience.
        < 2 records → defaults (0.08, 0.45). Loosens max_mean_shift when observations are noisy.
        """
        if len(self.experience) < 2:
            return {'max_mean_shift': 0.08, 'min_flip_precision': 0.45}
        past_ms = [r['actual']['actual_mean_shift'] for r in self.experience]
        ms_std  = float(np.std(past_ms))
        ms_mean = float(np.mean(past_ms))
        max_ms  = float(np.clip(ms_mean + 1.5 * ms_std, 0.06, 0.15))
        return {'max_mean_shift': max_ms, 'min_flip_precision': 0.45}

    def propose_counter_strategy(self, curr_result, prev_strategy) -> dict:
        """
        Memory-grounded rule-based counter-proposal.
        Halves grades that are consistently hurting; nudges up grades consistently helping.
        Used internally by arbitrate() as the LLM-failure fallback.
        """
        gw     = dict(prev_strategy.get('grade_weights', {}))
        curr_d = {g: float(s['delta']) for g, s in curr_result.get('grade_auc', {}).items()}
        past_d = {g: [] for g in gw}
        for rec in self.experience[-3:]:
            for g, d in rec['actual'].get('actual_per_grade_delta', {}).items():
                if g in past_d:
                    past_d[g].append(float(d))

        new_w = {}
        for g, w in gw.items():
            this   = curr_d.get(g, 0.0)
            recent = past_d.get(g, [])
            samples = recent + ([this] if this != 0 or recent else [])
            avg    = float(np.mean(samples)) if samples else 0.0
            if avg < -0.002 and this < 0:
                new_w[g] = round(w * 0.5, 3)
            elif avg > 0.005 and this > 0:
                new_w[g] = round(min(w + 0.02, 0.20), 3)
            else:
                new_w[g] = w
        return {
            'grade_weights':  new_w,
            'conf_threshold': prev_strategy.get('conf_threshold', 0.65),
            'source':         'green_counter',
        }

    def train_gating(self, subset_df, max_weight=0.6, l2=1e-3, max_iter=300,
                     random_state=42) -> dict:
        """
        Tier-A learned gating function. Trains
            g(features) → text_weight ∈ [0, max_weight]
        by minimizing log-loss of the fused prediction against the true label.

        Replaces the 5 grade_weights scalars with a per-row gate. Once set,
        evaluate_fusion uses self.gating_fn for text_weight regardless of
        strategy['grade_weights'].

        Features (per row):
            grade one-hot  (5)
            baseline_p
            text_confidence
            text_risk_score
            |text_risk_score - 0.5|        (text deviation from neutral)
            |baseline_p     - 0.5|        (baseline confidence)
            |text_risk_score - baseline_p| (disagreement magnitude)

        Model: w·standardize(features) + b → sigmoid → × max_weight.
        Optimizer: scipy L-BFGS-B with L2 regularization.

        Returns a diagnostic dict; also stored on self.gating_summary.
        """
        from scipy.optimize import minimize
        from sklearn.metrics import log_loss

        df_r = subset_df.reset_index(drop=True)
        n    = len(df_r)
        if n == 0 or 'label' not in df_r.columns:
            raise ValueError("train_gating: subset_df must contain a 'label' column and be non-empty")

        baseline_p = self.model.predict_proba(df_r[self.num_features].fillna(0))[:, 1]
        text_p     = df_r['text_risk_score'].values
        text_conf  = df_r['text_confidence'].values
        y          = df_r['label'].values.astype(float)

        grades = sorted(df_r['grade'].unique().tolist())
        grade_oh = np.zeros((n, len(grades)))
        for i, g in enumerate(grades):
            grade_oh[df_r['grade'].values == g, i] = 1.0

        nonbin = np.column_stack([
            baseline_p,
            text_conf,
            text_p,
            np.abs(text_p     - 0.5),
            np.abs(baseline_p - 0.5),
            np.abs(text_p - baseline_p),
        ])
        # Standardize non-one-hot features for optimization stability
        mu  = nonbin.mean(axis=0)
        std = nonbin.std(axis=0) + 1e-8
        nonbin_std = (nonbin - mu) / std

        feats = np.column_stack([grade_oh, nonbin_std])
        d     = feats.shape[1]

        base_logit = logit(baseline_p)
        text_ev    = logit(np.clip(text_p, 1e-6, 1 - 1e-6))  # logit(text) - logit(0.5) = logit(text)

        def predict_with_params(params):
            w, b = params[:-1], params[-1]
            gate_raw    = feats @ w + b
            text_weight = max_weight * (1.0 / (1.0 + np.exp(-gate_raw)))
            fused_logit = base_logit + text_weight * text_conf * text_ev
            fused_p     = 1.0 / (1.0 + np.exp(-fused_logit))
            return text_weight, np.clip(fused_p, 1e-7, 1 - 1e-7)

        def objective(params):
            _, fused_p = predict_with_params(params)
            ll  = -np.mean(y * np.log(fused_p) + (1 - y) * np.log(1 - fused_p))
            reg = l2 * np.sum(params[:-1] ** 2)
            return ll + reg

        ll_before = log_loss(y, np.clip(baseline_p, 1e-7, 1 - 1e-7))

        rng  = np.random.default_rng(random_state)
        init = rng.standard_normal(d + 1) * 0.1
        res  = minimize(objective, init, method='L-BFGS-B',
                        options={'maxiter': max_iter, 'gtol': 1e-6})

        w_opt = res.x[:-1].copy()
        b_opt = float(res.x[-1])
        tw_train, fp_after = predict_with_params(res.x)
        ll_after = log_loss(y, fp_after)

        feature_names = [f'grade_{g}' for g in grades] + [
            'baseline_p', 'text_conf', 'text_risk',
            'abs_text_dev', 'abs_base_dev', 'abs_diff',
        ]
        nonbin_start = len(grades)
        num_features = self.num_features  # capture for closure

        def gate_predict(df_predict):
            nr   = len(df_predict)
            bp   = self.model.predict_proba(df_predict[num_features].fillna(0))[:, 1]
            tp_v = df_predict['text_risk_score'].values
            tc_v = df_predict['text_confidence'].values
            g_oh = np.zeros((nr, len(grades)))
            for i, g in enumerate(grades):
                g_oh[df_predict['grade'].values == g, i] = 1.0
            nb_v = np.column_stack([
                bp, tc_v, tp_v,
                np.abs(tp_v - 0.5), np.abs(bp - 0.5), np.abs(tp_v - bp),
            ])
            nb_v = (nb_v - mu) / std
            ff   = np.column_stack([g_oh, nb_v])
            gate_raw = ff @ w_opt + b_opt
            return max_weight * (1.0 / (1.0 + np.exp(-gate_raw)))

        self.gating_fn      = gate_predict
        self.gating_summary = {
            'n_train':         n,
            'n_features':      d,
            'log_loss_before': float(ll_before),
            'log_loss_after':  float(ll_after),
            'improvement':     float(ll_before - ll_after),
            'converged':       bool(res.success),
            'iters':           int(res.nit),
            'feature_names':   feature_names,
            'coefficients':    dict(zip(feature_names, w_opt.tolist())),
            'bias':            b_opt,
            'max_weight':      max_weight,
            'mean_text_weight': float(tw_train.mean()),
            'std_text_weight':  float(tw_train.std()),
            # Serializable gate parameters for portability (e.g. demo pkl)
            'grades':          list(grades),
            'mu':              mu.tolist(),
            'std':             std.tolist(),
            'w':               w_opt.tolist(),
            'b':               float(b_opt),
        }
        return self.gating_summary

    def arbitrate(self, strategist_proposal, advocate_critique,
                  curr_result, prev_strategy, iteration) -> dict:
        """
        Active arbitration role — replaces the standalone arbitrator_agent.
        Green synthesizes a compromise between Strategist's proposal and
        Advocate's veto, informed by its own Reflexion experience.

        On LLM/JSON failure, falls back to propose_counter_strategy (memory-aware
        rule-based counter). Returns the same shape the old arbitrator returned,
        plus a 'source' field for downstream logging.
        """
        grade_lines = [
            f"  Grade {g}: AUPRC {s.get('baseline_auprc', 0):.3f}→{s.get('fused_auprc', 0):.3f} "
            f"(ΔAUPRC={s.get('delta_auprc', 0):+.4f}), AUC {s['baseline']:.3f}→{s['fused']:.3f}"
            for g, s in sorted(curr_result['grade_auc'].items())
        ]
        prompt = (
            f"You are the Green Agent acting as arbitrator — iteration {iteration}.\n"
            f"Synthesize a compromise between Strategist's proposal and Advocate's veto, "
            f"informed by your own prediction history.\n\n"
            f"Strategist proposed:\n"
            f"  grade_weights: {json.dumps(strategist_proposal.get('grade_weights', {}))}\n"
            f"  conf_threshold: {strategist_proposal.get('conf_threshold', 0.65)}\n"
            f"  rationale: {strategist_proposal.get('rationale', '')}\n\n"
            f"Advocate vetoed:\n"
            f"  {advocate_critique.get('critique', '')}\n"
            f"  Constraints: {'; '.join(advocate_critique.get('constraints', []))}\n\n"
            f"Current per-grade performance:\n" + "\n".join(grade_lines) + "\n\n"
            f"Previous strategy:\n"
            f"  grade_weights: {json.dumps(prev_strategy.get('grade_weights', {}))}\n"
            f"  conf_threshold: {prev_strategy.get('conf_threshold', 0.65)}\n\n"
            f"Your own prediction trajectory (Reflexion):\n{self._summarize_experience(3)}\n\n"
            "Decide weights that:\n"
            "  - reduce (don't zero) the affected grades from the Advocate's critique\n"
            "  - preserve overall AUC where Strategist sees clear gains\n"
            "  - if your past predictions have consistently overshot ΔAUC, lean conservative\n\n"
            "Weights in [0.0, 0.6], conf_threshold in [0.55, 0.85].\n"
            "CRITICAL OUTPUT FORMAT: respond with ONLY the JSON object. No reasoning, no markdown code fences, no commentary. Your response must start with { and end with }. Any text outside the JSON braces will break the parser.\n\n" + "Output ONLY a JSON object starting with { and ending with }:\n"
            '{"grade_weights": {"C": <float>, "D": <float>, "E": <float>, '
            '"F": <float>, "G": <float>}, '
            '"conf_threshold": <float>, "resolution": "<one sentence>"}'
        )

        try:
            raw   = call_llm(prompt, max_tokens=800, temperature=0.1)
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if not match:
                raise ValueError(f"no JSON in response: {raw[:200]}")
            result = json.loads(match.group(0))
            result['grade_weights'] = {
                g: round(max(0.0, min(0.6, float(w))), 3)
                for g, w in result.get('grade_weights', {}).items()
            }
            result['conf_threshold'] = round(
                max(0.55, min(0.85, float(result.get('conf_threshold', 0.65)))), 2)
            result['source']         = 'green_arbitrate'
            result['llm_called']     = True
            return result
        except Exception as e:
            fb = self.propose_counter_strategy(curr_result, prev_strategy)
            fb['resolution'] = f"LLM arbitrate failed ({e}); memory-based counter"
            fb['source']     = 'green_arbitrate_fallback'
            fb['llm_called'] = False
            return fb
    # ── End Reflexion infrastructure ──────────────────────────────────────

    def run_diagnostics(self, test_df, y_test) -> dict:
        preds = self.model.predict_proba(test_df[self.num_features].fillna(0))[:, 1]
        diag  = test_df.copy()
        diag['_pred']  = preds
        diag['_label'] = y_test.values
        diag['_error'] = ((preds > 0.5).astype(int) != y_test.values).astype(int)
        overall_error  = diag['_error'].mean()

        grade_stats = {}
        for grade, grp in diag.groupby('grade'):
            y_g = grp['_label']
            auc = round(float(roc_auc_score(y_g, grp['_pred'])), 3) if y_g.nunique() > 1 else float('nan')
            grade_stats[grade] = {
                'n':            len(grp),
                'default_rate': round(float(y_g.mean()), 3),
                'error_rate':   round(float(grp['_error'].mean()), 3),
                'auc':          auc,
            }
        high_error = [
            g for g, s in grade_stats.items()
            if not np.isnan(s['error_rate'])
            and s['error_rate'] - overall_error > 0.05
            and s['n'] >= 10
        ]
        return {
            'baseline_preds':    preds,
            'overall_auc':       round(float(roc_auc_score(y_test, preds)), 4),
            'overall_error':     round(float(overall_error), 3),
            'grade_stats':       grade_stats,
            'high_error_grades': high_error,
            'interruption':      f"Subgroup error spike: {high_error}" if high_error else None,
        }

    def evaluate_fusion(self, subset_df, strategy) -> dict:
        """
        Evaluates a fusion strategy on subset_df.
        Returns per-grade AUC, precision, and recall — the single source of truth
        for all White Agents (Advocate, Reporter) and Green's own arbitration role.
        """
        from sklearn.metrics import (precision_score as ps, recall_score as rs,
                                       average_precision_score as _aps)

        grade_weights  = strategy.get('grade_weights', {})
        conf_threshold = strategy.get('conf_threshold', 0.65)

        df_r           = subset_df.reset_index(drop=True)
        baseline_preds = self.model.predict_proba(df_r[self.num_features].fillna(0))[:, 1]

        if self.gating_fn is not None:
            # Tier-A: learned per-row gate overrides strategy['grade_weights']
            text_weight = self.gating_fn(df_r).astype(float).copy()
        else:
            text_weight = df_r['grade'].map(grade_weights).fillna(0.0).values.astype(float).copy()
        text_weight[df_r['text_confidence'].values < conf_threshold] = 0.0

        # Logit-space fusion: fused_logit = logit(baseline) + α × conf × [logit(text) − logit(0.5)]
        text_evidence = logit(df_r['text_risk_score'].values) - logit(0.5)
        fused_logit   = logit(baseline_preds) + text_weight * df_r['text_confidence'].values * text_evidence
        fused_preds   = np.where(text_weight == 0, baseline_preds, sigmoid(fused_logit))
        y_true      = df_r['label'].values

        grade_auc = {}
        for grade in sorted(df_r['grade'].unique()):
            mask = (df_r['grade'] == grade).values
            y_g  = y_true[mask]
            if mask.sum() < 10 or len(np.unique(y_g)) < 2:
                continue
            b      = roc_auc_score(y_g, baseline_preds[mask])
            f      = roc_auc_score(y_g, fused_preds[mask])
            ba     = float(_aps(y_g, baseline_preds[mask]))
            fa     = float(_aps(y_g, fused_preds[mask]))
            f_lbl  = (fused_preds[mask] > 0.5).astype(int)
            grade_auc[grade] = {
                'baseline':       round(b, 4),
                'fused':          round(f, 4),
                'delta':          round(f - b, 4),
                'baseline_auprc': round(ba, 4),
                'fused_auprc':    round(fa, 4),
                'delta_auprc':    round(fa - ba, 4),
                'precision':      round(ps(y_g, f_lbl, zero_division=0), 4),
                'recall':         round(rs(y_g, f_lbl, zero_division=0), 4),
                'n':              int(mask.sum()),
            }

        return {
            'baseline_preds':       baseline_preds,
            'fused_preds':          fused_preds,
            'overall_baseline_auc':   round(float(roc_auc_score(y_true, baseline_preds)), 4),
            'overall_fused_auc':      round(float(roc_auc_score(y_true, fused_preds)), 4),
            'overall_baseline_auprc': round(float(_aps(y_true, baseline_preds)), 4),
            'overall_fused_auprc':    round(float(_aps(y_true, fused_preds)), 4),
            'grade_auc':              grade_auc,
            'strategy_applied':       strategy,
        }


    def compute_flip_stats(self, baseline_preds, fused_preds, y_true, threshold=0.5) -> dict:
        """
        Analyzes verdict flips caused by text fusion.
        raised  = baseline says no-default, fused says default
        lowered = baseline says default,    fused says no-default
        """
        b = (baseline_preds >= threshold).astype(int)
        f = (fused_preds    >= threshold).astype(int)
        flipped = b != f
        stats   = {'n_flips': int(flipped.sum()),
                   'flip_rate': round(float(flipped.mean()), 4)}
        raised  = flipped & (f > b)
        lowered = flipped & (f < b)
        if raised.sum()  > 0:
            stats['raised_n']         = int(raised.sum())
            stats['raised_precision'] = round(float(y_true[raised].mean()), 4)
        if lowered.sum() > 0:
            stats['lowered_n']        = int(lowered.sum())
            stats['lowered_recall']   = round(float((1 - y_true[lowered]).mean()), 4)
        return stats

    def validate_strategy(self, subset_df, strategy,
                          max_mean_shift=None, min_flip_precision=None) -> dict:
        """
        Green's independent multi-criteria validation of a proposed fusion strategy.
        Checks: (1) shift magnitude, (2) raised-verdict precision, (3) lowered-verdict recall.
        When thresholds are None, adapts them from self.experience (Reflexion).
        """
        if max_mean_shift is None or min_flip_precision is None:
            adaptive = self.adaptive_validation_thresholds()
            if max_mean_shift     is None: max_mean_shift     = adaptive['max_mean_shift']
            if min_flip_precision is None: min_flip_precision = adaptive['min_flip_precision']

        df_r     = subset_df.reset_index(drop=True)
        baseline = self.model.predict_proba(df_r[self.num_features].fillna(0))[:, 1]
        gw       = strategy.get('grade_weights', {})
        thr      = strategy.get('conf_threshold', 0.65)
        if self.gating_fn is not None:
            w = self.gating_fn(df_r).astype(float).copy()
        else:
            w = df_r['grade'].map(gw).fillna(0.0).values.astype(float)
        w[df_r['text_confidence'].values < thr] = 0.0
        text_ev  = logit(df_r['text_risk_score'].values) - logit(0.5)
        f_logit  = logit(baseline) + w * df_r['text_confidence'].values * text_ev
        fused    = np.where(w == 0, baseline, sigmoid(f_logit))
        y        = df_r['label'].values

        issues      = []
        passed      = True
        mean_shift  = float(np.abs(fused - baseline).mean())

        if mean_shift > max_mean_shift:
            issues.append(f"Mean shift {mean_shift:.3f} > {max_mean_shift:.3f}: text dominating numeric model")
            passed = False

        flip_stats = self.compute_flip_stats(baseline, fused, y)
        if flip_stats['n_flips'] > 0:
            rp = flip_stats.get('raised_precision', 1.0)
            lr = flip_stats.get('lowered_recall',   1.0)
            if rp < min_flip_precision:
                issues.append(f"Raised-flip precision {rp:.3f} < {min_flip_precision}: too many false alarms")
                passed = False
            if lr < min_flip_precision:
                issues.append(f"Lowered-flip recall {lr:.3f} < {min_flip_precision}: too many missed defaults")
                passed = False

        return {'passed': passed, 'issues': issues,
                'mean_shift': round(mean_shift, 4), 'flip_stats': flip_stats,
                'thresholds_used': {'max_mean_shift': max_mean_shift,
                                    'min_flip_precision': min_flip_precision}}

    def evaluate_candidates(self, subset_df, strategies: list) -> list:
        """
        Evaluates candidate strategies with multi-criteria ranking.
        Primary:   strategies that pass Green validation (shift + flip quality)
        Secondary: overall fused AUC among passing strategies
        Failing strategies are ranked last regardless of AUC.
        """
        results = []
        for strat in strategies:
            r   = self.evaluate_fusion(subset_df, strat)
            val = self.validate_strategy(subset_df, strat)
            results.append({
                'strategy':     strat,
                'fused_auc':    r['overall_fused_auc'],
                'baseline_auc': r['overall_baseline_auc'],
                'grade_auc':    r['grade_auc'],
                'result':       r,
                'passed':       val['passed'],
                'mean_shift':   val['mean_shift'],
                'flip_stats':   val['flip_stats'],
                'issues':       val['issues'],
            })
        # Passing strategies ranked by AUC first; failing strategies ranked last
        return sorted(results, key=lambda x: (not x['passed'], -x['fused_auc']))


    def green_llm_review(self, candidate_results: list, subset_df, iteration: int) -> dict:
        """
        Called when evaluate_candidates finds no passing strategy.
        Green LLM reviews the least-bad option and decides:
          accept     — use the best-failing strategy anyway (with warning)
          adjust     — halve weights for failing grades
          safe_reset — reset all weights to a conservative baseline
        With Reflexion, the experience trajectory is injected into the prompt so
        the LLM can lean on its own past prediction errors.
        Returns a strategy dict with Green's final decision.
        """
        best = candidate_results[0]
        issues_summary = "; ".join(best.get("issues", ["unknown issue"]))
        flip_stats     = best.get("flip_stats", {})
        grade_deltas   = {g: round(s["delta"], 4)
                          for g, s in best.get("grade_auc", {}).items()}

        best_auprc = best.get('result', {}).get('overall_fused_auprc',
                       best.get('fused_auprc', float('nan')))
        prompt = (
            f"You are the Green Agent — iteration {iteration}.\n"
            "evaluate_candidates found NO strategy that passed all validation criteria.\n"
            "You must make a terminal decision on how to proceed.\n\n"
            f"Your prediction trajectory so far:\n{self._summarize_experience(3)}\n\n"
            f"Best candidate strategy: {json.dumps(best['strategy'].get('grade_weights', {}))}\n"
            f"Its fused AUPRC: {best_auprc:.4f}  (fused AUC: {best['fused_auc']:.4f})\n"
            f"Validation issues: {issues_summary}\n"
            f"Mean shift from baseline: {best.get('mean_shift', 0):.4f}\n"
            f"Flip stats: {json.dumps(flip_stats)}\n"
            f"Per-grade AUC deltas: {json.dumps(grade_deltas)}\n\n"
            "Choose one action:\n"
            "  accept     — accept the best-failing strategy (AUC improvement justifies the risk)\n"
            "  adjust     — halve weights for grades with negative or zero delta\n"
            "  safe_reset — reset all weights to 0.05 (conservative)\n\n"
            "Rules:\n"
            "- If mean_shift > 0.12 → prefer safe_reset\n"
            "- If fused AUPRC > baseline AUPRC by > 0.005 → consider accept\n"
            "- If flip stats show poor precision (< 0.40) → prefer adjust or safe_reset\n"
            "- If your past predictions consistently overshot ΔAUC, lean conservative\n\n"
            "CRITICAL OUTPUT FORMAT: respond with ONLY the JSON object. No reasoning, no markdown code fences, no commentary. Your response must start with { and end with }. Any text outside the JSON braces will break the parser.\n\n" + 'Output only JSON: {"action": "<accept|adjust|safe_reset>", "reasoning": "<one sentence>"}'
        )
        raw = call_llm(prompt, max_tokens=800, temperature=0.2)
        cleaned = re.sub(r"^```(?:json)?\s*", "", raw.strip())
        cleaned = re.sub(r"\s*```$", "", cleaned.strip())
        try:
            result = json.loads(re.search(r"\{.*\}", cleaned, re.DOTALL).group(0))
        except Exception as e:
            result = {"action": "safe_reset", "reasoning": f"[parse error: {e}]"}

        action = result.get("action", "safe_reset")
        prev_w = best["strategy"]["grade_weights"]

        if action == "accept":
            final_strategy = best["strategy"]
        elif action == "adjust":
            new_w = {g: (round(w * 0.5, 3) if grade_deltas.get(g, 0) <= 0 else w)
                     for g, w in prev_w.items()}
            final_strategy = {**best["strategy"], "grade_weights": new_w}
        else:  # safe_reset
            final_strategy = {**best["strategy"],
                               "grade_weights": {g: 0.05 for g in prev_w}}

        return {
            "strategy":  final_strategy,
            "action":    action,
            "reasoning": result.get("reasoning", ""),
            "llm_raw":   raw,
        }

    def check_psi(self, train_df, target_df, bins=10) -> dict:
        psi_results = {}
        for feat in self.num_features:
            try:
                t_vals = train_df[feat].dropna().values
                g_vals = target_df[feat].dropna().values
                bps    = np.unique(np.percentile(t_vals, np.linspace(0, 100, bins + 1)))
                if len(bps) < 3:
                    continue
                t_pct = np.histogram(t_vals, bins=bps)[0].astype(float)
                g_pct = np.histogram(g_vals, bins=bps)[0].astype(float)
                t_pct = t_pct / t_pct.sum() + 1e-8
                g_pct = g_pct / g_pct.sum() + 1e-8
                psi_results[feat] = round(float(np.sum((g_pct - t_pct) * np.log(g_pct / t_pct))), 4)
            except Exception:
                continue
        flagged = {f: v for f, v in psi_results.items() if v > 0.25}
        return {
            'psi_by_feature':   psi_results,
            'flagged_features': flagged,
            'interruption':     f"PSI drift detected: {list(flagged.keys())}" if flagged else None,
        }

    def find_threshold(self, preds, y_true, min_precision=0.40) -> dict:
        """
        Search [0.05, 0.95] for lowest threshold meeting min_precision with highest recall.
        Call on validation data only — never on test set.
        """
        from sklearn.metrics import recall_score
        best = {'threshold': 0.5, 'precision': 0.0, 'recall': 0.0}
        for t in [i / 100 for i in range(5, 96)]:
            pl   = (np.array(preds) >= t).astype(int)
            if pl.sum() == 0:
                continue
            prec = precision_score(y_true, pl, zero_division=0)
            rec  = recall_score(y_true, pl, zero_division=0)
            if prec >= min_precision and rec > best['recall']:
                best = {'threshold': t, 'precision': round(prec, 3), 'recall': round(rec, 3)}
        return best

    def enforce_threshold(self, preds, y_true, threshold=0.5, min_precision=0.40) -> dict:
        """Final precision gate. White Agents cannot override."""
        pred_labels = (np.array(preds) >= threshold).astype(int)
        if pred_labels.sum() == 0:
            return {'passed': False, 'precision': 0.0, 'reason': 'No positive predictions'}
        prec   = precision_score(y_true, pred_labels, zero_division=0)
        passed = bool(prec >= min_precision)
        return {
            'passed':        passed,
            'precision':     round(float(prec), 3),
            'threshold':     threshold,
            'min_precision': min_precision,
            'reason':        None if passed else f"Precision {prec:.3f} < min {min_precision}",
        }


#### White Agent 2 — Feature Strategist
*LLM called only when grade-level signals are ambiguous (mixed deltas or |delta| < 0.005)*

In [ ]:
def _feature_strategist_rule_fallback(green_result, prev_strategy, iteration,
                                      error=None) -> dict:
    """
    Original rule-based update, kept as fallback when the LLM Strategist
    is unavailable or fails to return valid JSON.
    """
    grade_auc = green_result['grade_auc']
    cur_thr   = prev_strategy.get('conf_threshold', 0.65)
    new_w = {}
    for grade in ['C', 'D', 'E', 'F', 'G']:
        w     = prev_strategy['grade_weights'].get(grade, 0.0)
        delta = grade_auc.get(grade, {}).get('delta', 0)
        if delta >= 0.005:
            new_w[grade] = round(min(w + 0.05, 0.6), 3)
        elif delta <= -0.005:
            new_w[grade] = round(max(w - 0.05, 0.0), 3)
        else:
            new_w[grade] = w
    return {
        'grade_weights':           new_w,
        'conf_threshold':          cur_thr,
        'per_grade_classification': {},
        'rationale':               f'rule fallback (LLM failed: {error})' if error else 'rule fallback',
        'llm_called':              False,
        'fallback_reason':         error or 'no LLM available',
    }


def feature_strategist_agent(green_result: dict, text_stats: dict,
                              prev_strategy: dict, iteration: int,
                              green=None, advocate_opportunities=None) -> dict:
    """
    LLM-driven Strategy proposer.

    Synthesizes:
      - Per-grade snapshot from Green's evaluate_fusion (deltas, prec, rec, n).
      - Per-grade text-vs-default calibration from text_stats.
      - Recent trajectory of actual per-grade deltas from green.experience.
      - Opportunity flags surfaced by the previous iteration's Advocate
        (grades with consistently positive delta worth nudging up).

    Drops the legacy rule-based fast path — LLM is ALWAYS the proposer.
    On LLM failure or JSON parse error, falls back to the original ±0.05 rule.

    Output shape (backwards-compatible with the old function):
      {'grade_weights', 'conf_threshold', 'rationale', 'llm_called',
       'per_grade_classification' (new), 'fallback_reason' (when fallback fired)}
    """
    grade_auc = green_result['grade_auc']
    cur_thr   = prev_strategy.get('conf_threshold', 0.65)

    # Multi-step trajectory from Green's Reflexion memory
    trajectory_lines = []
    if green is not None and getattr(green, 'experience', None):
        for rec in green.experience[-3:]:
            per_grade = rec['actual'].get('actual_per_grade_delta', {})
            if per_grade:
                fields = ", ".join(f"{g}: {d:+.4f}" for g, d in sorted(per_grade.items()))
                trajectory_lines.append(f"  iter {rec['iter']}: {fields}")

    # Current snapshot with text calibration context
    grade_lines = []
    for grade, s in sorted(grade_auc.items()):
        ts = text_stats.get(grade, {})
        grade_lines.append(
            f"  Grade {grade}: AUPRC {s.get('baseline_auprc', 0):.3f}→"
            f"{s.get('fused_auprc', 0):.3f} (ΔAUPRC={s.get('delta_auprc', 0):+.4f}), "
            f"AUC {s['baseline']:.3f}→{s['fused']:.3f} (ΔAUC={s['delta']:+.4f}), "
            f"prec={s.get('precision', 0):.3f}, rec={s.get('recall', 0):.3f}, n={s.get('n', '?')} | "
            f"avg_text_risk={ts.get('avg_risk_score', 'n/a')}, "
            f"actual_default={ts.get('actual_default_rate', 'n/a')}, "
            f"avg_conf={ts.get('avg_confidence', 'n/a')}"
        )

    opp_block = ""
    if advocate_opportunities:
        opp_block = (
            "\nAdvocate flagged opportunities from the previous iteration "
            "(not vetoes — grades whose pattern suggests their weight may be raised):\n"
            "  " + "\n  ".join(advocate_opportunities) + "\n"
        )

    prompt = (
        f"You are the Feature Strategist for a loan-default fusion system — iteration {iteration}.\n"
        "Propose updated grade_weights and conf_threshold based on multi-step evidence, "
        "not just the latest iteration.\n\n"
        "Current per-grade snapshot:\n" + "\n".join(grade_lines) + "\n\n"
        f"Recent trajectory (per-grade actual delta over last 3 iters from Green's memory):\n"
        + ("\n".join(trajectory_lines) if trajectory_lines else "  (no history yet — exploratory iteration)")
        + f"{opp_block}"
        f"\nPrevious strategy:\n"
        f"  grade_weights: {json.dumps(prev_strategy.get('grade_weights', {}))}\n"
        f"  conf_threshold: {cur_thr}\n\n"
        "Reasoning steps:\n"
        "  1. For each grade, classify the signal using the trajectory + current snapshot:\n"
        "     - PERSISTENT_WIN   (positive delta consistent across ≥2 recent iters)\n"
        "     - PERSISTENT_LOSS  (negative delta consistent across ≥2 recent iters)\n"
        "     - NOISY            (alternating signs or magnitude < 0.005)\n"
        "     - UNKNOWN          (insufficient history)\n"
        "  2. Decide per-grade weight adjustment:\n"
        "     - PERSISTENT_WIN  → raise weight by 0.03–0.05\n"
        "     - PERSISTENT_LOSS → lower weight by 0.03–0.05\n"
        "     - NOISY           → hold the weight, or tiny nudge of ±0.01 toward the prevailing sign\n"
        "     - UNKNOWN         → small exploratory move if this iter's delta is positive\n"
        "  3. Calibration sanity: if a grade's avg_text_risk diverges sharply from actual_default_rate, "
        "lean conservative on that grade's weight (text signal is mis-calibrated there).\n"
        "  4. conf_threshold: raise it if low-confidence text scores are degrading specific grades; "
        "lower it if high-confidence scores are reliably helpful.\n"
        "  5. Honor any Advocate opportunity flags by considering modest weight increases on those grades.\n\n"
        "Constraints:\n"
        "  - Weights in [0.0, 0.6], conf_threshold in [0.55, 0.85].\n"
        "  - Per-iter weight change must be ≤ 0.10 in magnitude (avoid over-correction).\n\n"
        "CRITICAL OUTPUT FORMAT: respond with ONLY the JSON object. No reasoning, no markdown code fences, no commentary. Your response must start with { and end with }. Any text outside the JSON braces will break the parser.\n\n" + "Output ONLY a JSON object starting with { and ending with }:\n"
        '{"grade_weights": {"C": <float>, "D": <float>, "E": <float>, "F": <float>, "G": <float>}, '
        '"conf_threshold": <float>, '
        '"per_grade_classification": {"C": "<label>", "D": "<label>", "E": "<label>", '
        '"F": "<label>", "G": "<label>"}, '
        '"rationale": "<one short sentence>"}'
    )

    try:
        raw   = call_llm(prompt, max_tokens=800, temperature=0.1)
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if not match:
            raise ValueError(f"no JSON in response: {raw[:200]}")
        result = json.loads(match.group(0))
        # Clamp weights and conf_threshold to valid ranges
        result['grade_weights'] = {
            g: round(max(0.0, min(0.6, float(w))), 3)
            for g, w in result.get('grade_weights', {}).items()
        }
        result['conf_threshold'] = round(
            max(0.55, min(0.85, float(result.get('conf_threshold', cur_thr)))), 2)
        # Cap per-iter weight change to ±0.10 (no LLM over-correction)
        for g, new_w in list(result['grade_weights'].items()):
            prev_w = prev_strategy.get('grade_weights', {}).get(g, 0.0)
            if abs(new_w - prev_w) > 0.10:
                result['grade_weights'][g] = round(
                    prev_w + (0.10 if new_w > prev_w else -0.10), 3)
        # Normalize classification map
        result['per_grade_classification'] = {
            str(g): str(c) for g, c in result.get('per_grade_classification', {}).items()
        }
        result['rationale']  = str(result.get('rationale', ''))
        result['llm_called'] = True
        result['llm_raw']    = raw
        return result
    except Exception as e:
        return _feature_strategist_rule_fallback(green_result, prev_strategy,
                                                  iteration, error=str(e))


#### White Agent 3 — Subgroup Advocate
*Rule-based only — no LLM needed; quantitative checks on AUC, precision, recall per grade*

In [ ]:
def _subgroup_advocate_rule_fallback(prev_result, curr_result,
                                     veto_threshold=0.02, error=None) -> dict:
    """
    Original rule-based veto logic, kept as fallback when the LLM Advocate
    is unavailable or fails to return valid JSON.
    """
    degraded = []
    for grade, curr in curr_result['grade_auc'].items():
        if grade not in prev_result['grade_auc']:
            continue
        prev = prev_result['grade_auc'][grade]
        auc_drop  = prev['fused']            - curr['fused']
        prec_drop = prev.get('precision', 0) - curr.get('precision', 0)
        issues = []
        if auc_drop  > veto_threshold: issues.append(f"AUC -{auc_drop:.3f}")
        if prec_drop > 0.05:           issues.append(f"precision -{prec_drop:.3f}")
        if issues:
            degraded.append({
                'grade': grade, 'auc_drop': round(auc_drop, 4),
                'prec_drop': round(prec_drop, 4),
                'recall': curr.get('recall', float('nan')), 'issues': issues,
            })
    if not degraded:
        return {'veto': False, 'affected_grades': [], 'critique': None,
                'constraints': [], 'opportunity_flags': [],
                'llm_called': False, 'fallback_reason': error or 'no LLM available'}
    affected    = [d['grade'] for d in degraded]
    constraints = [f"Grade {d['grade']}: reduce text weight ({', '.join(d['issues'])})"
                   for d in degraded]
    critique = (f"Strategy harms Grade(s) {', '.join(affected)}: "
                + "; ".join(f"Grade {d['grade']} {' + '.join(d['issues'])}" for d in degraded))
    return {
        'veto': True, 'affected_grades': affected,
        'critique': critique, 'constraints': constraints,
        'opportunity_flags': [], 'degraded_details': degraded,
        'llm_called': False, 'fallback_reason': error or 'no LLM available',
    }


def subgroup_advocate_agent(prev_result, curr_result, green=None,
                            veto_threshold=0.02) -> dict:
    """
    LLM-based Subgroup Advocate. Reviews per-grade trajectory + sample sizes
    to distinguish PERSISTENT harm from SAMPLING NOISE.

    Inputs:
      - prev_result, curr_result: Green.evaluate_fusion outputs from t-1 and t.
      - green: GreenAgent instance — provides .experience for multi-step trajectory.
      - veto_threshold: legacy threshold, used only by the rule-based fallback.

    Output (JSON-ish dict):
      {'veto', 'affected_grades', 'critique', 'constraints',
       'opportunity_flags', 'reasoning', 'llm_called'}

    Falls back to the rule-based version when prev_result is None,
    LLM is unavailable, or JSON parsing fails.
    """
    if prev_result is None:
        return {'veto': False, 'affected_grades': [], 'critique': None,
                'constraints': [], 'opportunity_flags': [], 'llm_called': False}

    # Build the trajectory snippet from Green's Reflexion memory
    trajectory_lines = []
    if green is not None and getattr(green, 'experience', None):
        for rec in green.experience[-3:]:
            per_grade = rec['actual'].get('actual_per_grade_delta', {})
            if per_grade:
                fields = ", ".join(f"{g}: Δ={d:+.4f}" for g, d in sorted(per_grade.items()))
                trajectory_lines.append(f"  iter {rec['iter']}: {fields}")

    # Current per-grade state
    grade_lines = [
        f"  Grade {g}: AUPRC {s.get('baseline_auprc', 0):.3f}→{s.get('fused_auprc', 0):.3f} "
        f"(ΔAUPRC={s.get('delta_auprc', 0):+.4f}), "
        f"AUC {s['baseline']:.3f}→{s['fused']:.3f} (ΔAUC={s['delta']:+.4f}), "
        f"prec={s['precision']:.3f}, rec={s['recall']:.3f}, n={s.get('n', '?')}"
        for g, s in sorted(curr_result['grade_auc'].items())
    ]

    # Single-step changes prev → curr
    diff_lines = []
    for g, curr in sorted(curr_result['grade_auc'].items()):
        if g in prev_result['grade_auc']:
            prev = prev_result['grade_auc'][g]
            diff_lines.append(
                f"  Grade {g}: AUPRC {prev.get('fused_auprc', 0):.3f} → "
                f"{curr.get('fused_auprc', 0):.3f} (step ΔAUPRC="
                f"{curr.get('fused_auprc', 0) - prev.get('fused_auprc', 0):+.4f}), "
                f"prec {prev.get('precision', 0):.3f} → {curr.get('precision', 0):.3f}"
            )

    prompt = (
        "You are the Subgroup Advocate in a loan-default fusion system.\n"
        "Your job: protect subgroups (loan grades) from being silently harmed by strategy changes, "
        "BUT distinguish persistent harm from sampling noise.\n\n"
        f"Current per-grade state:\n" + "\n".join(grade_lines) + "\n\n"
        f"Single-step changes (prev → curr):\n" + "\n".join(diff_lines) + "\n\n"
        f"Recent trajectory of actual per-grade deltas (Green's Reflexion memory):\n"
        + ("\n".join(trajectory_lines) if trajectory_lines else "  (no history yet)") + "\n\n"
        "Decision principles:\n"
        "  1. Veto ONLY if harm is PERSISTENT (negative delta in ≥2 recent iters for that grade) "
        "OR the single-step drop clearly exceeds sampling noise.\n"
        "  2. Account for sample size: small n means high noise — require larger drops to veto.\n"
        "     Heuristic: a drop of 0.02 on n=500 is meaningful; 0.02 on n=30 is mostly noise.\n"
        "  3. Flag OPPORTUNITIES (do NOT veto): grades with consistently positive delta that may "
        "deserve higher weight in the next strategy.\n"
        "  4. Be conservative about vetoing — false vetoes block real gains. Prefer flagging over vetoing.\n\n"
        "CRITICAL OUTPUT FORMAT: respond with ONLY the JSON object. No reasoning, no markdown code fences, no commentary. Your response must start with { and end with }. Any text outside the JSON braces will break the parser.\n\n" + "Output ONLY a JSON object starting with { and ending with }:\n"
        '{"veto": true|false, '
        '"affected_grades": [<grade>, ...], '
        '"critique": "<short summary or empty string>", '
        '"constraints": ["<one per affected grade>", ...], '
        '"opportunity_flags": ["<grade + observed pattern>", ...], '
        '"reasoning": "<one-sentence justification>"}'
    )

    try:
        raw   = call_llm(prompt, max_tokens=800, temperature=0.1)
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if not match:
            raise ValueError(f"no JSON in response: {raw[:200]}")
        parsed = json.loads(match.group(0))
        return {
            'veto':              bool(parsed.get('veto', False)),
            'affected_grades':   [str(g) for g in parsed.get('affected_grades', [])],
            'critique':          str(parsed.get('critique', '')),
            'constraints':       [str(c) for c in parsed.get('constraints', [])],
            'opportunity_flags': [str(f) for f in parsed.get('opportunity_flags', [])],
            'reasoning':         str(parsed.get('reasoning', '')),
            'llm_called':        True,
            'llm_raw':           raw,
        }
    except Exception as e:
        return _subgroup_advocate_rule_fallback(prev_result, curr_result,
                                                veto_threshold, error=str(e))


#### Iterative Loop Orchestrator
*Strategist → Advocate → Arbitrator (on conflict) → repeat*

In [ ]:
def run_iterative_loop(green, subset_df, text_stats, max_iter=5, veto_threshold=0.02):
    """
    Strategist proposes → Advocate (LLM) checks → Green arbitrates (on veto) and verifies.
    The standalone Arbitrator agent has been merged into GreenAgent.arbitrate().

    Reflexion: each iteration Green generates an expectation BEFORE evaluate_fusion and
    reflects on the actual outcome AFTER. The accumulated experience drives
      (a) adaptive validation thresholds (validate_strategy without explicit bounds),
      (b) Green's arbitration on veto (LLM with trajectory injected; falls back to
          memory-based propose_counter_strategy if the LLM call fails),
      (c) the LLM Advocate, which sees the same trajectory and distinguishes
          persistent harm from sampling noise (falls back to rule on LLM failure),
      (d) the green_llm_review prompt (trajectory injected as context).
    """
    strategy = {
        'grade_weights':  {'C': 0.03, 'D': 0.10, 'E': 0.05, 'F': 0.03, 'G': 0.03},
        'conf_threshold': 0.65,
    }
    decision_log               = []
    prev_result                = None
    llm_calls                  = 0
    last_advocate_opportunities = None  # carried from t-1 into Strategist at t

    for i in range(max_iter):
        print(f"\n{'='*65}")
        print(f"Iter {i}  weights={strategy['grade_weights']}  conf={strategy['conf_threshold']}")

        # ── Reflexion step 1: generate expectation BEFORE evaluating ──────
        expectation = green.generate_expectation(strategy, prev_result, i)
        if expectation.get('llm_called'):
            llm_calls += 1
        print(f"  [GREEN EXPECT] ΔAUC≈{expectation['expected_overall_delta_auc']:+.4f}  "
              f"mean_shift≈{expectation['expected_mean_shift']:.3f}  "
              f"conf={expectation['confidence']:.2f}  "
              f"({'LLM' if expectation.get('llm_called') else 'heuristic'})")

        curr_result = green.evaluate_fusion(subset_df, strategy)
        print(f"  overall  AUPRC: {curr_result['overall_baseline_auprc']:.4f} → "
              f"{curr_result['overall_fused_auprc']:.4f}  "
              f"(ΔAUPRC={curr_result['overall_fused_auprc']-curr_result['overall_baseline_auprc']:+.4f}, "
              f"AUC={curr_result['overall_baseline_auc']:.4f}→{curr_result['overall_fused_auc']:.4f})")
        for grade, s in sorted(curr_result['grade_auc'].items()):
            arrow_p = '↑' if s.get('delta_auprc', 0) > 0 else '↓'
            print(f"  Grade {grade}: AUPRC {s.get('baseline_auprc', 0):.3f} → "
                  f"{s.get('fused_auprc', 0):.3f} {arrow_p}{abs(s.get('delta_auprc', 0)):.3f}  "
                  f"(AUC {s['baseline']:.3f}→{s['fused']:.3f})  "
                  f"prec={s['precision']:.3f} rec={s['recall']:.3f}")

        # ── Reflexion step 2: reflect on expected vs actual ───────────────
        reflection = green.reflect(expectation, curr_result, strategy)
        adapt = green.adaptive_validation_thresholds()
        print(f"  [GREEN REFLECT] err ΔAUC={reflection['errors']['err_delta_auc']:+.4f}  "
              f"err mean_shift={reflection['errors']['err_mean_shift']:+.4f}  "
              f"surprise={reflection['errors']['surprise']:.3f}  "
              f"| adaptive max_mean_shift={adapt['max_mean_shift']:.3f}")

        advocate = subgroup_advocate_agent(prev_result, curr_result, green, veto_threshold)
        if advocate.get('llm_called'):
            llm_calls += 1
        adv_mode = 'LLM' if advocate.get('llm_called') else 'rule'
        if advocate.get('opportunity_flags'):
            print(f"  [ADVOCATE OPPORTUNITY] {'; '.join(advocate['opportunity_flags'])}")
        log_entry = {
            'iteration':                i,
            'strategy':                 strategy.copy(),
            'overall_baseline':         curr_result['overall_baseline_auc'],
            'overall_fused':            curr_result['overall_fused_auc'],
            'overall_baseline_auprc':   curr_result['overall_baseline_auprc'],
            'overall_fused_auprc':      curr_result['overall_fused_auprc'],
            'grade_auc':                curr_result['grade_auc'],
            'veto':             advocate['veto'],
            'veto_grades':      advocate['affected_grades'],
            'advocate_mode':    adv_mode,
            'advocate_reasoning':       advocate.get('reasoning', ''),
            'advocate_opportunities':   advocate.get('opportunity_flags', []),
            'expectation':      {
                'expected_delta_auc':  expectation['expected_overall_delta_auc'],
                'expected_mean_shift': expectation['expected_mean_shift'],
                'confidence':          expectation['confidence'],
                'rationale':           expectation.get('rationale', ''),
            },
            'reflection':       {
                'err_delta_auc':  reflection['errors']['err_delta_auc'],
                'err_mean_shift': reflection['errors']['err_mean_shift'],
                'surprise':       reflection['errors']['surprise'],
            },
        }

        if advocate['veto']:
            print(f"  [ADVOCATE VETO/{adv_mode}] {advocate['critique']}")
            if advocate.get('reasoning'):
                print(f"  [ADVOCATE REASON] {advocate['reasoning']}")
            try:
                strat_prop = feature_strategist_agent(
                    curr_result, text_stats, strategy, i,
                    green=green, advocate_opportunities=last_advocate_opportunities)
                llm_calls += 1 if strat_prop.get('llm_called') else 0
                cls = strat_prop.get('per_grade_classification', {})
                print(f"  [STRATEGIST/{('LLM' if strat_prop.get('llm_called') else 'rule')}] "
                      f"{strat_prop.get('rationale', '')}"
                      + (f"  cls={cls}" if cls else ""))

                # Green arbitrates directly (replaces standalone arbitrator_agent)
                arb = green.arbitrate(strat_prop, advocate, curr_result, strategy, i)
                if arb.get('llm_called'):
                    llm_calls += 1
                print(f"  [GREEN ARBITRATE] {arb.get('resolution', '')}  "
                      f"weights={arb.get('grade_weights', {})}  "
                      f"({'LLM' if arb.get('llm_called') else 'fallback'})")

                # Halve-affected numeric fallback as evaluate_candidates competitor
                fallback_w = {g: (round(w * 0.5, 3) if g in advocate['affected_grades'] else w)
                              for g, w in strategy['grade_weights'].items()}
                fallback   = {'grade_weights':  fallback_w,
                              'conf_threshold': strategy['conf_threshold'],
                              'source':         'fallback'}
                candidates = [arb, fallback]

                # Green verifies: green_arbitrate vs fallback — pick higher fused AUC among passing
                ranked = green.evaluate_candidates(subset_df, candidates)
                if not ranked[0]['passed']:
                    print(f"  [GREEN LLM] No passing strategy — calling Green LLM review...")
                    green_review = green.green_llm_review(ranked, subset_df, i)
                    llm_calls += 1
                    strategy = green_review['strategy']
                    source   = f"green_llm:{green_review['action']}"
                    print(f"  [GREEN LLM] {green_review['action']} — {green_review['reasoning']}")
                else:
                    winner   = ranked[0]
                    strategy = winner['strategy']
                    source   = winner['strategy'].get('source', 'unknown')
                    runner_up_auc = ranked[1]['fused_auc'] if len(ranked) > 1 else float('nan')
                    print(f"  → Green verified: {source} wins "
                          f"(AUC={winner['fused_auc']:.4f} vs next={runner_up_auc:.4f})")

                log_entry['event']                     = source
                log_entry['strategist_rationale']      = strat_prop.get('rationale', '')
                log_entry['strategist_classification'] = strat_prop.get('per_grade_classification', {})
                log_entry['arbitrate_resolution']      = arb.get('resolution', '')
                log_entry['arbitrate_source']          = arb.get('source', '')
            except Exception as e:
                print(f"  [GREEN ARBITRATE ERROR] {e} — halving affected weights")
                for grade in advocate['affected_grades']:
                    if grade in strategy['grade_weights']:
                        strategy['grade_weights'][grade] = round(
                            strategy['grade_weights'][grade] * 0.5, 3)
                log_entry['event'] = 'veto_fallback'

            log_entry['veto_critique']   = advocate['critique']
            decision_log.append(log_entry)
            prev_result                  = curr_result
            last_advocate_opportunities  = advocate.get('opportunity_flags') or None
            continue

        # Convergence: only after a genuine strategy update or arbitration
        last_event = decision_log[-1].get('event', '') if decision_log else ''
        if (prev_result is not None
                and last_event in ('strategy_update', 'green_arbitrate',
                                   'green_arbitrate_fallback', 'fallback')
                and abs(curr_result['overall_fused_auprc'] - prev_result['overall_fused_auprc']) < 0.002):
            log_entry['event'] = 'converged'
            decision_log.append(log_entry)
            print(f"  Converged at iteration {i}.")
            break

        # No veto — Strategist proposes directly
        try:
            new_strat  = feature_strategist_agent(
                curr_result, text_stats, strategy, i,
                green=green, advocate_opportunities=last_advocate_opportunities)
            llm_called = new_strat.get('llm_called', True)
            llm_calls += 1 if llm_called else 0
            new_strategy = {
                'grade_weights':  new_strat['grade_weights'],
                'conf_threshold': new_strat.get('conf_threshold', strategy['conf_threshold']),
            }
            if (new_strategy['grade_weights'] == strategy['grade_weights'] and
                    new_strategy['conf_threshold'] == strategy['conf_threshold']):
                event_label = 'strategy_unchanged'
            else:
                event_label = 'strategy_update'
            strategy = new_strategy
            cls = new_strat.get('per_grade_classification', {})
            print(f"  [STRATEGIST/{('LLM' if llm_called else 'rule')}] "
                  f"{new_strat.get('rationale', '')} [{event_label}]"
                  + (f"  cls={cls}" if cls else ""))
            log_entry['event']                       = event_label
            log_entry['strategist_rationale']        = new_strat.get('rationale', '')
            log_entry['strategist_classification']   = cls
            log_entry['llm_called']                  = llm_called
        except Exception as e:
            print(f"  [STRATEGIST ERROR] {e} — keeping strategy")
            log_entry['event'] = 'strategist_error'

        decision_log.append(log_entry)
        prev_result                  = curr_result
        last_advocate_opportunities  = advocate.get('opportunity_flags') or None

    print(f"\nLLM calls during loop: {llm_calls}")
    print(f"Green experience records: {len(green.experience)}  "
          f"(mean |ΔAUC err|={np.mean([abs(r['errors']['err_delta_auc']) for r in green.experience]):.4f}, "
          f"mean |mean_shift err|={np.mean([abs(r['errors']['err_mean_shift']) for r in green.experience]):.4f})")
    return decision_log, curr_result


#### White Agent 5 — Reporter
*LLM called per prediction when baseline is in fuzzy zone [0.30–0.65] or text significantly shifted the outcome*

In [118]:
def reporter_agent(row: pd.Series, baseline_pred: float, fused_pred: float,
                   strategy: dict) -> dict:
    """
    Generates a faithful, auditable explanation for a single prediction.
    Flags fragile predictions where text overrode a confident numeric signal.
    Called for fuzzy-zone cases (baseline 0.30-0.65) or when text shifted prediction > 0.10.
    """
    eff_weight     = strategy['grade_weights'].get(row['grade'], 0.0)
    conf           = row.get('text_confidence', 0)
    if conf < strategy.get('conf_threshold', 0.65):
        eff_weight = 0.0

    text_influence = fused_pred - baseline_pred
    in_fuzzy_zone  = 0.30 <= baseline_pred <= 0.65
    text_flipped   = (fused_pred > 0.5) != (baseline_pred > 0.5)
    fragile        = text_flipped and abs(text_influence) < 0.15

    prompt = (
        f"You are explaining a loan default prediction to an auditor.\n\n"
        f"Borrower description: {str(row.get('desc', ''))[:400]}\n\n"
        f"Numeric model (XGBoost) prediction: {baseline_pred:.3f}\n"
        f"Text risk score: {row.get('text_risk_score', 'N/A')}\n"
        f"Text confidence: {conf:.3f}\n"
        f"Effective text weight for Grade {row['grade']}: {eff_weight:.2f}\n"
        f"Final fused prediction: {fused_pred:.3f}  (text shifted by {text_influence:+.3f})\n\n"
        f"Key numeric features: int_rate={row.get('int_rate','N/A')}, "
        f"dti={row.get('dti','N/A')}, fico={row.get('fico_range_low','N/A')}, "
        f"annual_inc={row.get('annual_inc','N/A')}\n\n"
        'Output a single JSON object and nothing else:\n'
        '{"explanation": "2-3 sentences describing why this prediction was made",'
        ' "main_driver": "text | numeric | both",'
        ' "faithfulness": "was the prediction mainly moved by text or numeric features?",'
        ' "fragility_flag": true/false,'
        ' "fragility_reason": "null or brief explanation"}'
    )

    resp = requests.post(
        MINIMAX_URL,
        headers={"Authorization": f"Bearer {MINIMAX_API_KEY}", "Content-Type": "application/json"},
        json={"model": "MiniMax-Text-01",
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": 300, "temperature": 0.3},
        timeout=30
    )
    raw = resp.json()['choices'][0]['message']['content']
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {'explanation': 'N/A', 'fragility_flag': fragile,
                'fragility_reason': 'computed', '_error': raw[:100]}
    result = json.loads(match.group(0))
    result['_text_influence'] = round(text_influence, 4)
    result['_in_fuzzy_zone']  = in_fuzzy_zone
    result['_fragile_computed'] = fragile
    return result

#### Run the Full System

In [ ]:
# ── Train XGBoost baseline ────────────────────────────────────────────
model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    scale_pos_weight=(train['label'] == 0).sum() / (train['label'] == 1).sum(),
    random_state=42, eval_metric='auc'
)
model.fit(train[NUM_FEATURES], train['label'])
print(f"Baseline AUC (full test): "
      f"{roc_auc_score(test['label'], model.predict_proba(test[NUM_FEATURES].fillna(0))[:,1]):.4f}")

green = GreenAgent(model, NUM_FEATURES)

baseline_diag = green.run_diagnostics(test, test['label'])
if baseline_diag['interruption']:
    print(f"[GREEN INTERRUPT] {baseline_diag['interruption']}")

psi = green.check_psi(train, subset_train)
if psi['interruption']:
    print(f"[GREEN PSI] {psi['interruption']}")
    print(f"  (Note: C-G grades naturally differ from full train — some drift is expected)")
else:
    print("PSI check: no significant feature drift.")

# ── Iterative loop on TRAIN (tune weights, no test leakage) ───────────
print("\n--- Tuning fusion weights on TRAIN ---")
decision_log, train_result = run_iterative_loop(
    green, subset_train, text_stats, max_iter=5, veto_threshold=0.02
)
best_strategy = decision_log[-1]['strategy']
print(f"\nBest strategy: {best_strategy}")

# ── Decision threshold ───────────────────────────────────────────────────────
# Canonical decision threshold is 0.5 — same value used by evaluate_fusion,
# compute_flip_stats, validate_strategy, and every per-grade precision/recall.
# find_threshold below is INFORMATIONAL: it reports the threshold that WOULD be
# needed to enforce a precision floor, but does not change the operating point.
DECISION_THRESHOLD = 0.5
best_threshold     = DECISION_THRESHOLD

cal = green.find_threshold(train_result['fused_preds'], subset_train['label'].values,
                           min_precision=0.40)
print(f"\nDecision threshold (operating point): {DECISION_THRESHOLD}")
if cal['recall'] > 0:
    print(f"  [INFO] To enforce precision >= 0.40, threshold would need to be "
          f"{cal['threshold']} (train precision={cal['precision']}, recall={cal['recall']})")
else:
    print(f"  [INFO] No threshold in [0.05, 0.95] meets precision >= 0.40 on train")

# ── Final evaluation on TEST (held-out, run once) ─────────────────────
print("\n--- Final evaluation on TEST ---")
final_result = green.evaluate_fusion(subset_test, best_strategy)
print(f"Test baseline AUPRC : {final_result['overall_baseline_auprc']:.4f}  "
      f"(AUC {final_result['overall_baseline_auc']:.4f})")
print(f"Test fused   AUPRC  : {final_result['overall_fused_auprc']:.4f}  "
      f"(AUC {final_result['overall_fused_auc']:.4f})")
print(f"               ΔAUPRC: {final_result['overall_fused_auprc'] - final_result['overall_baseline_auprc']:+.4f}  "
      f"(ΔAUC {final_result['overall_fused_auc'] - final_result['overall_baseline_auc']:+.4f})")
for grade, s in sorted(final_result['grade_auc'].items()):
    arrow_p = '↑' if s.get('delta_auprc', 0) > 0 else '↓'
    print(f"  Grade {grade}: AUPRC {s.get('baseline_auprc', 0):.3f} → "
          f"{s.get('fused_auprc', 0):.3f} {arrow_p}{abs(s.get('delta_auprc', 0)):.3f}  "
          f"(AUC {s['baseline']:.3f}→{s['fused']:.3f})")



In [ ]:
# ── Save model artifacts to pkl ───────────────────────────────────────────────
import pickle, os

# Build grade_train_stats: per-grade AUC delta at learned weight and at α=0.20
grade_train_stats = {}
for grade in ['C', 'D', 'E', 'F', 'G']:
    learned_w = best_strategy['grade_weights'].get(grade, 0.0)
    s = train_result['grade_auc'].get(grade, {})
    delta_learned = s.get('delta', 0.0)

    # Measure AUC delta at high weight (α=0.20) for Advocate reference in demo
    high_strat = {**best_strategy, 'grade_weights': {**best_strategy['grade_weights'], grade: 0.20}}
    high_res   = green.evaluate_fusion(subset_train, high_strat)
    delta_high = high_res['grade_auc'].get(grade, {}).get('delta', 0.0)

    if delta_learned <= -0.005:
        reason = 'text fusion hurt AUC at any weight'
    elif delta_learned > 0.01:
        reason = 'text signal improves AUC'
    elif learned_w == 0.0:
        reason = 'insufficient samples or no signal'
    else:
        reason = 'marginal, small weight only'

    grade_train_stats[grade] = {
        'delta_at_learned_weight': round(delta_learned, 4),
        'delta_at_high_weight':    round(delta_high, 4),
        'learned_weight':          learned_w,
        'reason':                  reason,
    }

print("grade_train_stats:")
for g, v in grade_train_stats.items():
    print(f"  {g}: {v}")

# Save
save_path = "../models/loan_default_model.pkl"
with open(save_path, 'wb') as f:
    pickle.dump({
        'model':             model,
        'features':          NUM_FEATURES,
        'best_strategy':     best_strategy,
        'grade_norm_stats':  grade_norm_stats,
        'grade_train_stats': grade_train_stats,
    }, f)
print(f"\nSaved → {save_path}")


#### Comparison 0 — Tier-A Learned Gating Function

Replaces the 5 hand-tuned `grade_weights` scalars with a learned per-row gate
`g(grade, baseline_p, conf, text_risk, |text-0.5|, |baseline-0.5|, |text-baseline|) → text_weight`.
Trained by minimizing log-loss of the fused prediction on `subset_train`.


In [ ]:
# ── Tier-A: train learned per-row gating function ──────────────────────────
# Snapshot the current (Strategist-tuned) test result for side-by-side comparison.
import copy
from sklearn.metrics import roc_auc_score as _auc, log_loss as _ll, brier_score_loss as _brier

# Capture pre-gate results from the Strategist pipeline
pre_gate_summary = {
    'fused_auc_test':      final_result['overall_fused_auc'],
    'baseline_auc_test':   final_result['overall_baseline_auc'],
    'fused_auprc_test':    final_result['overall_fused_auprc'],
    'baseline_auprc_test': final_result['overall_baseline_auprc'],
    'log_loss_test':       float(_ll(subset_test['label'].values,
                                      np.clip(final_result['fused_preds'], 1e-7, 1-1e-7))),
    'brier_test':          float(_brier(subset_test['label'].values, final_result['fused_preds'])),
    'mean_text_weight_test': None,
}

# Train the gate on subset_train (same data Strategist saw)
gate_info = green.train_gating(subset_train, max_weight=0.6, l2=1e-3, max_iter=300)
print("=== Tier-A gating training ===")
print(f"  n_train:          {gate_info['n_train']}")
print(f"  converged:        {gate_info['converged']}  (iters={gate_info['iters']})")
print(f"  log_loss before:  {gate_info['log_loss_before']:.5f}  (baseline-only)")
print(f"  log_loss after:   {gate_info['log_loss_after']:.5f}  (fused with gate)")
print(f"  improvement:      {gate_info['improvement']:+.5f}")
print(f"  text_weight on TRAIN: mean={gate_info['mean_text_weight']:.3f}, "
      f"std={gate_info['std_text_weight']:.3f}")

# Top-magnitude coefficients (most-influential gate inputs)
coefs = sorted(gate_info['coefficients'].items(), key=lambda kv: -abs(kv[1]))
print("\n  top features by |coef| (post-standardization):")
for name, c in coefs[:6]:
    print(f"    {name:<14s} {c:+.3f}")
print(f"    bias           {gate_info['bias']:+.3f}")

# Re-evaluate test with the gate on
final_result_gate = green.evaluate_fusion(subset_test, best_strategy)
fp_gate           = final_result_gate['fused_preds']
y_test            = subset_test['label'].values

post_gate_summary = {
    'fused_auc_test':      final_result_gate['overall_fused_auc'],
    'baseline_auc_test':   final_result_gate['overall_baseline_auc'],
    'fused_auprc_test':    final_result_gate['overall_fused_auprc'],
    'baseline_auprc_test': final_result_gate['overall_baseline_auprc'],
    'log_loss_test':       float(_ll(y_test, np.clip(fp_gate, 1e-7, 1-1e-7))),
    'brier_test':          float(_brier(y_test, fp_gate)),
    'mean_text_weight_test': float(green.gating_fn(subset_test).mean()),
}

print("\n=== Side-by-side on TEST ===")
print(f"  {'metric':<24s} {'Strategist (5 scalars)':>22s}  {'Tier-A gate (learned)':>22s}  {'Δ':>9s}")
def _row(name, a, b, prec=4, hib='+', sign=True):
    da = b - a if (a is not None and b is not None) else None
    a_s = f"{a:.{prec}f}" if a is not None else "—"
    b_s = f"{b:.{prec}f}" if b is not None else "—"
    if da is None:           d_s = "—"
    elif sign:               d_s = f"{da:+.{prec}f}"
    else:                    d_s = f"{da:.{prec}f}"
    print(f"  {name:<24s} {a_s:>22s}  {b_s:>22s}  {d_s:>9s}")

_row('overall fused AUPRC',pre_gate_summary['fused_auprc_test'], post_gate_summary['fused_auprc_test'])
_row('overall fused AUC',  pre_gate_summary['fused_auc_test'],   post_gate_summary['fused_auc_test'])
_row('overall log-loss',   pre_gate_summary['log_loss_test'],    post_gate_summary['log_loss_test'])
_row('Brier score',        pre_gate_summary['brier_test'],       post_gate_summary['brier_test'])
_row('mean text_weight',   pre_gate_summary['mean_text_weight_test'], post_gate_summary['mean_text_weight_test'], prec=3)

# Per-grade comparison
print("\n=== Per-grade fused AUPRC ===")
print(f"  {'grade':<6} {'baseline':>9} {'strat':>9} {'gate':>9} {'gate-strat':>11}")
for g in sorted(set(final_result['grade_auc']) & set(final_result_gate['grade_auc'])):
    s_pre  = final_result['grade_auc'][g]
    s_post = final_result_gate['grade_auc'][g]
    bp     = s_pre.get('baseline_auprc', 0)
    sp     = s_pre.get('fused_auprc', 0)
    gp     = s_post.get('fused_auprc', 0)
    print(f"  {g:<6} {bp:>9.4f} {sp:>9.4f} {gp:>9.4f} {gp - sp:>+11.4f}")

# Toggle hint
print("\nTo revert to the 5-scalar strategist: `green.gating_fn = None`")
print("To re-train the gate: `green.train_gating(subset_train)`")


# ── Re-save pkl with the trained gate (so demo can use it) ──────────────────
import pickle
save_path = "../models/loan_default_model.pkl"
gate_payload = None
if green.gating_summary is not None:
    gs = green.gating_summary
    gate_payload = {
        'grades':     gs['grades'],
        'mu':         gs['mu'],
        'std':        gs['std'],
        'w':          gs['w'],
        'b':          gs['b'],
        'max_weight': gs['max_weight'],
        'feature_names': gs['feature_names'],
        'log_loss_improvement': gs['improvement'],
        'mean_text_weight_train': gs['mean_text_weight'],
    }
with open(save_path, 'wb') as f:
    pickle.dump({
        'model':             model,
        'features':          NUM_FEATURES,
        'best_strategy':     best_strategy,
        'grade_norm_stats':  grade_norm_stats,
        'grade_train_stats': grade_train_stats,
        'gating_params':     gate_payload,   # NEW — None if gate not trained
    }, f)
print(f"\nRe-saved pkl with Tier-A gate → {save_path}")
print(f"  gating_params present:        {gate_payload is not None}")


#### Trust the Text — Multi-Agent Evaluation Toolkit

Reusable metric functions that answer six questions about *when* and *whether* to trust the LLM text signal, comparing three pipelines side by side:

1. **baseline-only** (no text)
2. **Strategist** (5 LLM-tuned `grade_weights` scalars)
3. **Tier-A gate** (learned per-row gating function)

Each metric maps to one question; the final runner prints a coherent narrative.


In [ ]:
# ── "When to trust the text" — evaluation toolkit ──────────────────────────
from sklearn.metrics import (average_precision_score, brier_score_loss,
                              log_loss as _ll)

# ── Q1: Does the multi-agent fusion improve discrimination? ───────────────
def auprc_with_ci(y, p, n_bootstrap=200, seed=42):
    """AUPRC with percentile bootstrap CI. Reports point + 95% CI."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p)
    n = len(y); bs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        ys = y[idx]
        if ys.sum() == 0 or ys.sum() == n:
            continue
        bs.append(average_precision_score(ys, p[idx]))
    return {
        'point': float(average_precision_score(y, p)),
        'lo':    float(np.percentile(bs,  2.5)) if bs else float('nan'),
        'hi':    float(np.percentile(bs, 97.5)) if bs else float('nan'),
        'n_bootstrap': len(bs),
    }


# ── Q2: Does fusion break calibration? ─────────────────────────────────────
def brier_ece(y, p, n_bins=10):
    """Brier + Expected Calibration Error + per-bin reliability data."""
    y = np.asarray(y); p = np.asarray(p)
    brier  = float(np.mean((p - y) ** 2))
    edges  = np.linspace(0, 1, n_bins + 1)
    rows   = []
    ece    = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = ((p >= lo) & (p < hi)) if i < n_bins - 1 else ((p >= lo) & (p <= hi))
        if mask.sum() == 0:
            continue
        avg_p, avg_y = float(p[mask].mean()), float(y[mask].mean())
        ece += (mask.sum() / len(y)) * abs(avg_p - avg_y)
        rows.append({'bin': f"[{lo:.2f},{hi:.2f}]", 'n': int(mask.sum()),
                     'mean_pred': avg_p, 'frac_pos': avg_y,
                     'gap': avg_p - avg_y})
    return {'brier': brier, 'ece': float(ece), 'reliability_bins': rows}


# ── Q3: Is fusion net-beneficial across decision thresholds? ──────────────
def decision_curve(y, p, thresholds=None):
    """Net benefit at each threshold: NB(t) = TP/N − FP/N · t/(1−t).
    Returns the curve plus treat-all and treat-none baselines."""
    y = np.asarray(y); p = np.asarray(p)
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.50, 19)  # credit cutoffs typically low-side
    n      = len(y); prev = y.mean()
    rows   = []
    for t in thresholds:
        pred = (p >= t).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum())
        fp = int(((pred == 1) & (y == 0)).sum())
        nb        = (tp / n) - (fp / n) * (t / max(1 - t, 1e-6))
        treat_all = prev      - (1 - prev) * (t / max(1 - t, 1e-6))
        rows.append({'threshold': float(t), 'net_benefit': float(nb),
                     'treat_all': float(treat_all)})
    return rows


# ── Q4: When fusion changes the verdict, does it reclassify correctly? ────
def nri(y, p_old, p_new, threshold=0.5):
    """Categorical NRI at threshold + continuous (direction-only) NRI.

    NRI = P(↑ | event) − P(↓ | event) + P(↓ | non-event) − P(↑ | non-event)
    Positive = fusion is reclassifying in the right direction overall.
    """
    y = np.asarray(y); p_old = np.asarray(p_old); p_new = np.asarray(p_new)
    pos, neg = y == 1, y == 0
    cls_old = (p_old >= threshold).astype(int)
    cls_new = (p_new >= threshold).astype(int)
    up   = cls_new > cls_old
    down = cls_new < cls_old
    cat_nri = ((up[pos].sum() - down[pos].sum()) / max(pos.sum(), 1)
               + (down[neg].sum() - up[neg].sum()) / max(neg.sum(), 1))
    inc, dec = p_new > p_old, p_new < p_old
    cont_nri = ((inc[pos].sum() - dec[pos].sum()) / max(pos.sum(), 1)
                + (dec[neg].sum() - inc[neg].sum()) / max(neg.sum(), 1))
    return {
        'categorical_nri': float(cat_nri),
        'continuous_nri':  float(cont_nri),
        'up_events':       int(up[pos].sum()),
        'down_events':     int(down[pos].sum()),
        'up_nonevents':    int(up[neg].sum()),
        'down_nonevents':  int(down[neg].sum()),
    }


# ── Q5: Is the LLM's self-reported confidence a real trust signal? ────────
def llm_confidence_reliability(text_score, text_conf, y, n_bins=8):
    """Bin by LLM-reported confidence; report empirical accuracy of
    (text_score > 0.5 == label).

    If accuracy is flat across confidence bins → LLM's self-reported confidence
    carries no outcome information; the conf_threshold gate in evaluate_fusion
    is effectively a placebo.
    """
    text_score = np.asarray(text_score)
    text_conf  = np.asarray(text_conf)
    y          = np.asarray(y)
    pred       = (text_score > 0.5).astype(int)
    correct    = (pred == y).astype(int)
    edges      = np.linspace(text_conf.min(), text_conf.max(), n_bins + 1)
    rows       = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = ((text_conf >= lo) & (text_conf < hi)) if i < n_bins - 1 \
                else ((text_conf >= lo) & (text_conf <= hi))
        if mask.sum() < 5:
            continue
        rows.append({
            'bin':       f"[{lo:.2f},{hi:.2f}]",
            'n':         int(mask.sum()),
            'mean_conf': float(text_conf[mask].mean()),
            'accuracy':  float(correct[mask].mean()),
            'mean_text_risk': float(text_score[mask].mean()),
            'frac_default':   float(y[mask].mean()),
        })
    if len(rows) >= 2:
        confs = np.array([r['mean_conf'] for r in rows])
        accs  = np.array([r['accuracy']  for r in rows])
        slope = float(np.polyfit(confs, accs, 1)[0])
    else:
        slope = float('nan')
    return {'bins': rows, 'slope': slope}


# ── Q6: WHEN should we trust the text? (the headline question) ────────────
def selective_fusion_curve(y, p_baseline, p_fused, trust_score,
                            n_points=20, min_n=20):
    """Sweep coverage k% (top-k rows by `trust_score`); for each coverage,
    compute AUPRC of fused vs baseline on those rows.

    `trust_score` should be the per-row signal you'd use to decide whether
    to trust the text on this row. Typical choices:
      - text_confidence (LLM self-report)
      - |text_risk - baseline_p| (model disagreement)
      - learned gate output  (Tier-A)

    The curve's SHAPE is the answer to 'when to trust the text':
      - Monotonically decreasing → trust only at the very top (selective wins)
      - Inverted-U                → there's an optimal coverage τ*
      - Flat                      → trust score carries no useful signal
    """
    y = np.asarray(y); p_baseline = np.asarray(p_baseline)
    p_fused = np.asarray(p_fused); trust_score = np.asarray(trust_score)
    n     = len(y)
    order = np.argsort(-trust_score)   # high → low
    rows  = []
    for k in np.linspace(0.05, 1.0, n_points):
        cutoff = int(k * n)
        if cutoff < min_n:
            continue
        idx = order[:cutoff]
        yk  = y[idx]
        if yk.sum() == 0 or yk.sum() == cutoff:
            continue
        b   = average_precision_score(yk, p_baseline[idx])
        f   = average_precision_score(yk, p_fused[idx])
        rows.append({'coverage': float(k), 'n': cutoff,
                     'baseline_auprc': float(b), 'fused_auprc': float(f),
                     'delta_auprc':    float(f - b)})
    return rows


# ── Multi-agent report runner: weave the 6 metrics into a narrative ───────
def trust_the_text_report(green, subset, strategy, label_col='label',
                          bootstrap_n=200, plot=False):
    """
    Compare baseline / Strategist / Tier-A gate using the 6 metrics above and
    print a coherent narrative.

    Assumes:
      - green is a GreenAgent with a trained model
      - subset has columns: numeric features, grade, text_risk_score,
        text_confidence, and the label column
      - strategy is the final post-loop strategy (used for the 5-scalar pipeline)
      - If green.gating_fn is None, Tier-A is skipped
    """
    y          = subset[label_col].values.astype(int)
    text_score = subset['text_risk_score'].values
    text_conf  = subset['text_confidence'].values

    baseline_p = green.model.predict_proba(subset[green.num_features].fillna(0))[:, 1]

    # Toggle gate off → Strategist fusion
    saved_gate = green.gating_fn
    green.gating_fn = None
    fp_strat = green.evaluate_fusion(subset, strategy)['fused_preds']
    # Restore gate (if any) → Tier-A fusion
    green.gating_fn = saved_gate
    if saved_gate is not None:
        fp_gate = green.evaluate_fusion(subset, strategy)['fused_preds']
    else:
        fp_gate = None

    def _line():
        print("─" * 78)

    # ── Q1: AUPRC ──────────────────────────────────────────────────────────
    _line()
    print("Q1. Does fusion improve discrimination at this base rate?")
    print(f"    (positive rate = {y.mean():.3f} — AUC is too easy at this skew; AUPRC is the right lens)")
    for name, p in [('baseline', baseline_p),
                    ('Strategist (5 scalars)', fp_strat),
                    ('Tier-A gate', fp_gate)]:
        if p is None: continue
        a = auprc_with_ci(y, p, n_bootstrap=bootstrap_n)
        print(f"    {name:<22s}  AUPRC = {a['point']:.4f}  "
              f"[95% CI: {a['lo']:.4f}, {a['hi']:.4f}]")

    # ── Q2: Calibration ────────────────────────────────────────────────────
    _line()
    print("Q2. Does fusion stay calibrated? (lower Brier/ECE = better)")
    print(f"    {'pipeline':<22s}  {'Brier':>8s}  {'ECE':>8s}  {'log-loss':>8s}")
    for name, p in [('baseline', baseline_p),
                    ('Strategist (5 scalars)', fp_strat),
                    ('Tier-A gate', fp_gate)]:
        if p is None: continue
        be = brier_ece(y, p)
        ll = _ll(y, np.clip(p, 1e-7, 1-1e-7))
        print(f"    {name:<22s}  {be['brier']:>8.4f}  {be['ece']:>8.4f}  {ll:>8.4f}")

    # ── Q3: Decision curve / net benefit ──────────────────────────────────
    _line()
    print("Q3. Across plausible decision thresholds, is fusion net-beneficial?")
    print("    (showing net benefit at 3 thresholds; positive = better than treat-none)")
    print(f"    {'pipeline':<22s} {'t=0.10':>10s} {'t=0.20':>10s} {'t=0.30':>10s}")
    for name, p in [('baseline', baseline_p),
                    ('Strategist (5 scalars)', fp_strat),
                    ('Tier-A gate', fp_gate)]:
        if p is None: continue
        dc = decision_curve(y, p, thresholds=np.array([0.10, 0.20, 0.30]))
        vals = "  ".join(f"{r['net_benefit']:>+8.4f}" for r in dc)
        print(f"    {name:<22s} {vals}")

    # ── Q4: NRI — does fusion reclassify in the right direction? ───────────
    _line()
    print("Q4. When fusion changes the 0.5-threshold verdict, does it reclassify correctly?")
    print("    (NRI > 0 means more right-direction reclassifications than wrong)")
    r_base_strat = nri(y, baseline_p, fp_strat)
    print(f"    Strategist  vs  baseline   →  categorical NRI = {r_base_strat['categorical_nri']:+.4f}  "
          f"(continuous NRI = {r_base_strat['continuous_nri']:+.4f})")
    print(f"        events: up={r_base_strat['up_events']}, down={r_base_strat['down_events']}  | "
          f"non-events: up={r_base_strat['up_nonevents']}, down={r_base_strat['down_nonevents']}")
    if fp_gate is not None:
        r_strat_gate = nri(y, fp_strat, fp_gate)
        print(f"    Tier-A gate vs  Strategist →  categorical NRI = {r_strat_gate['categorical_nri']:+.4f}  "
              f"(continuous NRI = {r_strat_gate['continuous_nri']:+.4f})")
        print(f"        events: up={r_strat_gate['up_events']}, down={r_strat_gate['down_events']}  | "
              f"non-events: up={r_strat_gate['up_nonevents']}, down={r_strat_gate['down_nonevents']}")

    # ── Q5: LLM confidence reliability (the headline diagnostic) ──────────
    _line()
    print("Q5. Is the LLM Text Analyst's self-reported confidence informative?")
    print("    (if accuracy is flat across confidence bins → conf_threshold gate is placebo)")
    rel = llm_confidence_reliability(text_score, text_conf, y, n_bins=6)
    print(f"    {'bin':<14s} {'n':>5s} {'mean_conf':>10s} {'accuracy':>10s} {'mean_text':>10s} {'frac_dflt':>10s}")
    for r in rel['bins']:
        print(f"    {r['bin']:<14s} {r['n']:>5d} {r['mean_conf']:>10.3f} "
              f"{r['accuracy']:>10.3f} {r['mean_text_risk']:>10.3f} {r['frac_default']:>10.3f}")
    slope_msg = (f"    accuracy ~ confidence slope: {rel['slope']:+.3f}  "
                 + ("→ informative" if abs(rel['slope']) > 0.05
                    else "→ FLAT — LLM confidence not informative; conf_threshold gate is placebo"))
    print(slope_msg)

    # ── Q6: Selective fusion curve (the headline question) ───────────────
    _line()
    print("Q6. WHEN should we trust the text?  (selective fusion curves)")
    print("    For each candidate trust-score, sweep coverage from top-5% down to 100%,")
    print("    and ask: does the fused AUPRC beat baseline on the top-k? Best curve wins.")
    trust_signals = {
        'text_confidence (LLM self-report)': text_conf,
        '|text_risk - baseline| (disagreement)': np.abs(text_score - baseline_p),
    }
    if green.gating_fn is not None:
        trust_signals['Tier-A gate output'] = green.gating_fn(subset)

    # Pick the active fused (gate if available, else strategist)
    fp_for_select = fp_gate if fp_gate is not None else fp_strat
    fp_label      = 'Tier-A gate' if fp_gate is not None else 'Strategist'
    print(f"    (using {fp_label} as the fused pipeline)\n")
    print(f"    {'trust-score':<40s} {'cov=10%':>10s} {'cov=30%':>10s} {'cov=50%':>10s} {'cov=100%':>10s}")
    print(f"    {'(showing ΔAUPRC over baseline)':<40s}")
    for name, ts in trust_signals.items():
        curve = selective_fusion_curve(y, baseline_p, fp_for_select, ts, n_points=20)
        def _at(target):
            best = None
            for r in curve:
                if best is None or abs(r['coverage'] - target) < abs(best['coverage'] - target):
                    best = r
            return best['delta_auprc'] if best is not None else float('nan')
        d10, d30, d50, d100 = (_at(0.10), _at(0.30), _at(0.50), _at(1.00))
        print(f"    {name:<40s} {d10:>+10.4f} {d30:>+10.4f} {d50:>+10.4f} {d100:>+10.4f}")

    # ── Optional plotting ─────────────────────────────────────────────────
    if plot:
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            # Reliability
            ax = axes[0]
            confs = [r['mean_conf'] for r in rel['bins']]
            accs  = [r['accuracy']  for r in rel['bins']]
            ax.plot(confs, accs, 'o-', label='empirical')
            ax.plot([0, 1], [0.5, 0.5], 'k:', alpha=0.4, label='chance (0.5)')
            ax.set_xlabel("LLM-reported confidence")
            ax.set_ylabel("Text-only accuracy")
            ax.set_title("Q5: LLM confidence reliability")
            ax.legend()
            # Selective fusion curve (Tier-A gate trust-score if available)
            ax = axes[1]
            for name, ts in trust_signals.items():
                curve = selective_fusion_curve(y, baseline_p, fp_for_select, ts, n_points=20)
                xs = [r['coverage'] for r in curve]
                ys = [r['delta_auprc'] for r in curve]
                ax.plot(xs, ys, 'o-', label=name)
            ax.axhline(0, color='k', linestyle=':', alpha=0.4)
            ax.set_xlabel("Coverage (top-k by trust score)")
            ax.set_ylabel("ΔAUPRC (fused − baseline)")
            ax.set_title(f"Q6: Selective fusion curve ({fp_label})")
            ax.legend(fontsize=8)
            plt.tight_layout(); plt.show()
        except Exception as e:
            print(f"(plot skipped: {e})")

    _line()
    return {
        'baseline_p':  baseline_p,
        'strat_p':     fp_strat,
        'gate_p':      fp_gate,
        'reliability': rel,
    }


# Run the full report on the test set
_ = trust_the_text_report(green, subset_test, best_strategy, plot=True)


#### Decision Log

In [ ]:
rows = []
for e in decision_log:
    event = e.get('event', '-')
    rows.append({
        'iter':         e['iteration'],
        'event':        event,
        'baseline_auc': e['overall_baseline'],
        'fused_auc':    e['overall_fused'],
        'delta_auc':    round(e['overall_fused'] - e['overall_baseline'], 4),
        'baseline_auprc': e.get('overall_baseline_auprc', float('nan')),
        'fused_auprc':    e.get('overall_fused_auprc',    float('nan')),
        'delta_auprc':    round(e.get('overall_fused_auprc', 0)
                                  - e.get('overall_baseline_auprc', 0), 4),
        'veto':         e['veto'],
        'veto_grades':  (','.join(e['veto_grades']) if (e['veto'] and e['veto_grades']) else '-'),
        'adv':          e.get('advocate_mode', '-'),
        'llm':          ('yes' if e.get('llm_called')
                          else ('arb' if event in ('green_arbitrate', 'green_arbitrate_fallback')
                          else 'no')),
        'conf_thr':     e['strategy']['conf_threshold'],
        'w_C':          e['strategy']['grade_weights'].get('C', 0),
        'w_D':          e['strategy']['grade_weights'].get('D', 0),
        'w_E':          e['strategy']['grade_weights'].get('E', 0),
        'exp_dAUC':     round(e.get('expectation', {}).get('expected_delta_auc', 0.0), 4),
        'err_dAUC':     round(e.get('reflection',  {}).get('err_delta_auc',      0.0), 4),
        'opportunities': '; '.join(e.get('advocate_opportunities', []))[:50] or '-',
        'classification': ', '.join(f"{g}:{c[:4]}" for g, c in
                                    sorted((e.get('strategist_classification') or {}).items())) or '-',
        'note':         (e.get('arbitrate_resolution',
                          e.get('strategist_rationale',
                           e.get('veto_critique', '-'))) or '-')[:60],
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Per-grade trajectory (AUPRC primary, AUC in parens) ===")
grade_rows = []
for e in decision_log:
    for grade, s in e['grade_auc'].items():
        grade_rows.append({
            'iter':            e['iteration'],
            'grade':           grade,
            'baseline_auprc':  s.get('baseline_auprc', float('nan')),
            'fused_auprc':     s.get('fused_auprc',    float('nan')),
            'delta_auprc':     s.get('delta_auprc',    float('nan')),
            'baseline_auc':    s['baseline'],
            'fused_auc':       s['fused'],
            'delta_auc':       s['delta'],
            'n':               s.get('n', '?'),
        })
print(pd.DataFrame(grade_rows).to_string(index=False))


#### Per-Agent KPI — Milestone-Based Contribution Tracking

Implements the MultiAgentBench paper's core metric (§3.3):

$$\mathrm{KPI}_j = \frac{n_j}{M}, \quad \mathrm{KPI}_{\text{overall}} = \frac{1}{N}\sum_{j=1}^{N}\mathrm{KPI}_j$$

where $n_j$ is the number of milestones agent $j$ contributed to and $M$ is the
total milestone count. This decomposes the system's success into per-agent
load, surfacing **hub agents** and **bottleneck agents**.

Our schema: 7 per-iter milestones × `n_iters` + 4 once-per-run milestones, with
fixed agent-attribution rules (no LLM-based detection — deterministic).


In [ ]:
# ── Per-Agent KPI: MultiAgentBench §3.3 ──────────────────────────────────
# KPI_j = n_j / M,  Overall KPI = mean_j(KPI_j)
#   n_j  = milestones agent j contributed to
#   M    = total milestone slots = (per-iter count) × n_iters + once count

PER_ITER_MILESTONES = [
    {
        'name':        'M_expect_generated',
        'description': "Green generated an expectation before evaluate_fusion",
        'agents':      ['Green'],
        'check':       lambda e: 'expectation' in e,
    },
    {
        'name':        'M_low_surprise',
        'description': "Green's prediction error matched reality (surprise < 0.05)",
        'agents':      ['Green'],
        'check':       lambda e: e.get('reflection', {}).get('surprise', float('inf')) < 0.05,
    },
    {
        'name':        'M_strategy_proposed',
        'description': "Strategist produced a non-trivial proposal (won or vetoed)",
        'agents':      ['Strategist'],
        'check':       lambda e: (
            e.get('event') in ('strategy_update', 'fallback')
            or str(e.get('event', '')).startswith('green_arbitrate')
            or str(e.get('event', '')).startswith('green_llm:')
        ),
    },
    {
        'name':        'M_no_veto',
        'description': "Advocate let the strategy pass (subgroup-safe)",
        'agents':      ['Advocate'],
        'check':       lambda e: not e.get('veto', False),
    },
    {
        'name':        'M_opportunity_flagged',
        'description': "Advocate flagged ≥1 positive opportunity",
        'agents':      ['Advocate'],
        'check':       lambda e: bool(e.get('advocate_opportunities', [])),
    },
    {
        'name':        'M_arbitration_succeeded',
        'description': "On veto, Green's arbitrate resolved without LLM safety net",
        'agents':      ['Green', 'Strategist', 'Advocate'],
        'check':       lambda e: (
            e.get('veto', False)
            and str(e.get('event', '')).startswith('green_arbitrate')
        ),
    },
    {
        'name':        'M_auprc_improved_iter',
        'description': "Fused AUPRC exceeded baseline AUPRC this iteration",
        'agents':      ['Strategist', 'TextAnalyst'],
        'check':       lambda e: (e.get('overall_fused_auprc', 0)
                                   > e.get('overall_baseline_auprc', 0)),
    },
]

ONCE_MILESTONES = [
    {
        'name':        'M_loop_converged',
        'description': "Loop reached convergence before max_iter",
        'agents':      ['Green', 'Strategist', 'Advocate'],
        'check':       lambda log, fr, gr, ba: any(e.get('event') == 'converged' for e in log),
    },
    {
        'name':        'M_test_above_baseline',
        'description': "Final fused AUPRC on test > baseline AUPRC",
        'agents':      ['Strategist', 'TextAnalyst'],
        'check':       lambda log, fr, gr, ba: (fr.get('overall_fused_auprc', 0)
                                                 > fr.get('overall_baseline_auprc', 0)),
    },
    {
        'name':        'M_gate_trained',
        'description': "Tier-A gate was trained and the optimizer converged",
        'agents':      ['Green'],
        'check':       lambda log, fr, gr, ba: (
            getattr(gr, 'gating_summary', None) is not None
            and gr.gating_summary.get('converged', False)
        ),
    },
    {
        'name':        'M_gate_helps',
        'description': "Tier-A gate's training log-loss beat baseline-only log-loss",
        'agents':      ['Green', 'TextAnalyst'],
        'check':       lambda log, fr, gr, ba: (
            getattr(gr, 'gating_summary', None) is not None
            and gr.gating_summary['log_loss_after'] < gr.gating_summary['log_loss_before']
        ),
    },
]


def compute_per_agent_kpi(decision_log, final_result, green, baseline_auc,
                          all_agents=('TextAnalyst', 'Strategist', 'Advocate', 'Green')):
    """
    Compute per-agent KPI from a completed run.

    Returns dict with:
      milestones        — list of every milestone instance (with iter, name, agents, fired)
      total_milestones  — M (denominator)
      milestones_fired  — count of fired milestones
      kpi_by_agent      — {agent: {n, kpi, contributed_to}}
      overall_kpi       — mean of agent KPIs
      n_iters           — number of loop iterations
    """
    instances = []

    # Per-iter milestones
    for entry in decision_log:
        i = entry.get('iteration', '?')
        for m in PER_ITER_MILESTONES:
            try:
                ok = bool(m['check'](entry))
            except Exception:
                ok = False
            instances.append({
                'iter':   i,
                'name':   m['name'],
                'agents': m['agents'],
                'fired':  ok,
            })

    # Once-per-run milestones
    for m in ONCE_MILESTONES:
        try:
            ok = bool(m['check'](decision_log, final_result, green, baseline_auc))
        except Exception:
            ok = False
        instances.append({
            'iter':   None,
            'name':   m['name'],
            'agents': m['agents'],
            'fired':  ok,
        })

    total_M       = len(instances)
    fired_count   = sum(1 for inst in instances if inst['fired'])
    by_agent      = {a: {'n': 0, 'contributed_to': []} for a in all_agents}

    for inst in instances:
        if not inst['fired']:
            continue
        tag = (f"iter{inst['iter']}:{inst['name']}"
               if inst['iter'] is not None else inst['name'])
        for a in inst['agents']:
            if a in by_agent:
                by_agent[a]['n'] += 1
                by_agent[a]['contributed_to'].append(tag)

    for a in by_agent:
        by_agent[a]['kpi'] = by_agent[a]['n'] / total_M if total_M > 0 else 0.0

    overall_kpi = (sum(d['kpi'] for d in by_agent.values()) / len(by_agent)
                   if by_agent else 0.0)

    return {
        'milestones':       instances,
        'total_milestones': total_M,
        'milestones_fired': fired_count,
        'kpi_by_agent':     by_agent,
        'overall_kpi':      overall_kpi,
        'n_iters':          len(decision_log),
    }


def print_per_agent_kpi_report(kpi):
    """Pretty-print the per-agent KPI summary + fire matrix."""
    print("=" * 78)
    print(f"Per-Agent KPI Report")
    print(f"  Total milestones (M):    {kpi['total_milestones']}  "
          f"({kpi['milestones_fired']} fired, fire-rate "
          f"{kpi['milestones_fired'] / max(kpi['total_milestones'], 1):.3f})")
    print(f"  Loop iterations:         {kpi['n_iters']}")
    print(f"  Overall KPI:             {kpi['overall_kpi']:.3f}")
    print()
    print(f"  {'agent':<14s} {'n_j':>5s} {'KPI_j':>7s}   role-density: top contributions")
    for a, d in sorted(kpi['kpi_by_agent'].items(),
                       key=lambda kv: -kv[1]['kpi']):
        head = ", ".join(d['contributed_to'][:4])
        more = f" (+{len(d['contributed_to']) - 4} more)" if len(d['contributed_to']) > 4 else ""
        print(f"  {a:<14s} {d['n']:>5d} {d['kpi']:>7.3f}   {head}{more}")

    # Fire matrix: rows = milestones, cols = iters + once
    by_name = {}
    for inst in kpi['milestones']:
        by_name.setdefault(inst['name'], []).append(inst)

    print()
    print("  Milestone fire matrix  (✓ = fired, · = did not, — = N/A)")
    print(f"  {'milestone':<28s} {'per-iter':<15s} {'once':<6s} agents")
    print(f"  {'-' * 28} {'-' * 15} {'-' * 6} {'-' * 30}")
    for name, insts in by_name.items():
        per_iter = "".join("✓" if x['fired'] else "·"
                            for x in insts if x['iter'] is not None)
        once_mks = "".join("✓" if x['fired'] else "·"
                            for x in insts if x['iter'] is None)
        if not per_iter: per_iter = "—"
        if not once_mks: once_mks = "—"
        agents = ",".join(insts[0]['agents'])
        print(f"  {name:<28s} {per_iter:<15s} {once_mks:<6s} {agents}")
    print("=" * 78)


# Run on the completed loop
_kpi = compute_per_agent_kpi(
    decision_log, final_result, green,
    baseline_auc=baseline_diag['overall_auc'],
)
print_per_agent_kpi_report(_kpi)


#### Comparison 1 — Meta-Model (Text as Features)

In [121]:
from sklearn.linear_model import LogisticRegression

def add_text_flags(df):
    """Regex-based binary flags — no extra API calls needed."""
    d = df.copy()
    desc = d['desc'].fillna('').str.lower()
    d['flag_consolidation'] = desc.str.contains('consolidat').astype(float)
    d['flag_repayment_plan'] = desc.str.contains(r'will pay|plan to|repay|pay off', regex=True).astype(float)
    d['flag_stress']        = desc.str.contains(r'behind|urgent|emergency|struggling|desperate', regex=True).astype(float)
    d['flag_stable_income'] = desc.str.contains(r'stable|steady|permanent|full.time|full time', regex=True).astype(float)
    return d

META_FEATURES = NUM_FEATURES + ['text_risk_score', 'text_confidence',
                                  'flag_consolidation', 'flag_repayment_plan',
                                  'flag_stress', 'flag_stable_income']

tr = add_text_flags(subset_train)
te = add_text_flags(subset_test)

meta_model = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1)
meta_model.fit(tr[META_FEATURES].fillna(0), tr['label'])

meta_preds = meta_model.predict_proba(te[META_FEATURES].fillna(0))[:, 1]
print(f"Baseline (numeric only) AUC : {final_result['overall_baseline_auc']:.4f}")
print(f"Weighted fusion AUC          : {final_result['overall_fused_auc']:.4f}")
print(f"Meta-model (LR stacking) AUC : {roc_auc_score(subset_test['label'], meta_preds):.4f}")

# Show per-grade meta-model AUC
print("\nPer-grade meta-model AUC:")
for grade in sorted(te['grade'].unique()):
    mask = (te['grade'] == grade).values
    y_g  = subset_test['label'].values[mask]
    if mask.sum() < 10 or len(np.unique(y_g)) < 2:
        continue
    b = roc_auc_score(y_g, final_result['baseline_preds'][mask])
    m = roc_auc_score(y_g, meta_preds[mask])
    print(f"  Grade {grade}: baseline={b:.3f}  meta={m:.3f}  delta={m-b:+.3f}")

Baseline (numeric only) AUC : 0.6444
Weighted fusion AUC          : 0.6444
Meta-model (LR stacking) AUC : 0.5733

Per-grade meta-model AUC:
  Grade C: baseline=0.738  meta=0.750  delta=+0.012
  Grade D: baseline=0.667  meta=0.458  delta=-0.208


/Users/ljw/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### Comparison 2 — Grid Search vs Strategist

In [122]:
import itertools

# Grid search on TRAIN subset — all combinations
cands_c   = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
cands_d   = [0.0, 0.05, 0.1, 0.15, 0.2]
cands_e   = [0.0, 0.05, 0.1]
cands_thr = [0.55, 0.60, 0.65, 0.70, 0.75]

best_grid_auc  = -1
best_grid_strat = None

for w_c, w_d, w_e, thr in itertools.product(cands_c, cands_d, cands_e, cands_thr):
    strat = {'grade_weights': {'C': w_c, 'D': w_d, 'E': w_e, 'F': 0.0, 'G': 0.0},
             'conf_threshold': thr}
    res = green.evaluate_fusion(subset_train, strat)
    if res['overall_fused_auc'] > best_grid_auc:
        best_grid_auc   = res['overall_fused_auc']
        best_grid_strat = strat

print(f"Grid search best (train AUC={best_grid_auc:.4f}): {best_grid_strat}")

grid_test = green.evaluate_fusion(subset_test, best_grid_strat)
strat_test_auc = final_result['overall_fused_auc']

print(f"\nTest AUC comparison:")
print(f"  Baseline                  : {final_result['overall_baseline_auc']:.4f}")
print(f"  Strategist (MAS loop)     : {strat_test_auc:.4f}")
print(f"  Grid search (450 combos)  : {grid_test['overall_fused_auc']:.4f}")
print(f"  Strategist vs grid search : {strat_test_auc - grid_test['overall_fused_auc']:+.4f}")

Grid search best (train AUC=0.7473): {'grade_weights': {'C': 0.0, 'D': 0.2, 'E': 0.1, 'F': 0.0, 'G': 0.0}, 'conf_threshold': 0.6}

Test AUC comparison:
  Baseline                  : 0.6444
  Strategist (MAS loop)     : 0.6444
  Grid search (450 combos)  : 0.6178
  Strategist vs grid search : +0.0266


#### Live Walkthrough — Individual Case Analysis

In [123]:
def show_walkthrough(subset_df, green, final_strategy, n_cases=4, call_reporter=True):
    """
    Displays representative prediction cases.
    For each case, optionally calls Reporter Agent for faithful explanation + fragility audit.
    """
    df_w = subset_df.copy().reset_index(drop=True)
    df_w['baseline_pred'] = green.model.predict_proba(df_w[NUM_FEATURES].fillna(0))[:, 1]

    gw  = final_strategy['grade_weights']
    thr = final_strategy['conf_threshold']
    df_w['eff_weight'] = df_w['grade'].map(gw).fillna(0.0)
    df_w.loc[df_w['text_confidence'] < thr, 'eff_weight'] = 0.0
    # Logit-space fusion (mirrors evaluate_fusion)
    _text_ev = logit(df_w['text_risk_score'].values) - logit(0.5)
    _fused_logit = logit(df_w['baseline_pred'].values) + df_w['eff_weight'].values * df_w['text_confidence'].values * _text_ev
    df_w['fused_pred'] = np.where(df_w['eff_weight'] == 0, df_w['baseline_pred'],
                                  sigmoid(_fused_logit))
    df_w['baseline_ok'] = ((df_w['baseline_pred'] > 0.5) == df_w['label'])
    df_w['fused_ok']    = ((df_w['fused_pred']    > 0.5) == df_w['label'])
    df_w['disagree']    = abs(df_w['baseline_pred'] - df_w['text_risk_score'])
    df_w['in_fuzzy']    = (df_w['baseline_pred'].between(0.30, 0.65))

    case_pools = [
        ("Text HELPED  — baseline wrong, fusion right",
         df_w[~df_w['baseline_ok'] & df_w['fused_ok'] & (df_w['eff_weight'] > 0)]),
        ("Text HURT    — baseline right, fusion wrong",
         df_w[df_w['baseline_ok'] & ~df_w['fused_ok'] & (df_w['eff_weight'] > 0)]),
        ("FUZZY ZONE   — numeric model uncertain (baseline 0.30-0.65)",
         df_w[df_w['in_fuzzy'] & (df_w['eff_weight'] > 0)].nlargest(5, 'disagree')),
        ("Text SUPPRESSED — confidence below threshold",
         df_w[df_w['eff_weight'] == 0].nlargest(5, 'text_confidence')),
    ]

    shown = 0
    for case_label, pool in case_pools:
        if shown >= n_cases or len(pool) == 0:
            continue
        row = pool.iloc[0]
        shown += 1

        print(f"\n{'='*70}")
        print(f"CASE: {case_label}")
        print(f"Grade: {row['grade']}  |  Actual: {'DEFAULT' if row['label']==1 else 'FULLY PAID'}")
        print(f"{'─'*70}")
        print(f"Description:\n  {str(row['desc'])[:350]}")
        print(f"{'─'*70}")
        print(f"[Text Analyst]   risk_score={row['text_risk_score']:.3f}  "
              f"confidence={row['text_confidence']:.3f}")
        print(f"[Green Numeric]  baseline_pred={row['baseline_pred']:.3f}"
              f"{'  ← fuzzy zone' if row['in_fuzzy'] else ''}")
        print(f"[Fusion]         eff_weight={row['eff_weight']:.2f} (thr={thr})  "
              f"fused_pred={row['fused_pred']:.3f}  "
              f"(text shift {row['fused_pred']-row['baseline_pred']:+.3f})")
        print(f"[Outcome]        {'CORRECT ✓' if row['fused_ok'] else 'WRONG ✗'}")

        if call_reporter and (row['in_fuzzy'] or abs(row['fused_pred'] - row['baseline_pred']) > 0.10):
            try:
                report = reporter_agent(row, row['baseline_pred'], row['fused_pred'], final_strategy)
                print(f"[Reporter]")
                print(f"  Explanation : {report.get('explanation', 'N/A')}")
                print(f"  Main driver : {report.get('main_driver', 'N/A')}")
                print(f"  Faithfulness: {report.get('faithfulness', 'N/A')}")
                if report.get('fragility_flag') or report.get('_fragile_computed'):
                    reason = report.get('fragility_reason') or 'text flipped a borderline numeric prediction'
                    print(f"  ⚠ FRAGILE  : {reason}")
            except Exception as e:
                print(f"  [REPORTER ERROR] {e}")


show_walkthrough(subset_test, green, best_strategy)


CASE: FUZZY ZONE   — numeric model uncertain (baseline 0.30-0.65)
Grade: C  |  Actual: FULLY PAID
──────────────────────────────────────────────────────────────────────
Description:
    Borrower added on 11/07/10 > This will consolidate 4 credit cards into a lower monthly payment. Currently I Pay $250monthly to cover the minimum on these cards.<br/> Borrower added on 11/07/10 > Planning to set up automatic payments from checking to cover monthly payments. Also expect to have this paid off early.  One of my 2 jobs is Residential 
──────────────────────────────────────────────────────────────────────
[Text Analyst]   risk_score=0.201  confidence=0.950
[Green Numeric]  baseline_pred=0.429  ← fuzzy zone
[Fusion]         eff_weight=0.03 (thr=0.65)  fused_pred=0.420  (text shift -0.010)
[Outcome]        CORRECT ✓
[Reporter]
  Explanation : The prediction of a potential loan default is influenced by a moderate interest rate and debt-to-income ratio, combined with an average FICO score and re